# What Your CXR Reveals Beyond the Pathology
## Demographic Leakage in Medical Foundation Model Embeddings and Its Calibration Fairness Consequences

**ECCV 2026 Workshop on Medical Foundation Models and Benchmarks (MEDFMB) · Reproducibility Notebook · v4 — Two Datasets, Causal Ablation, Full Robustness Suite**

| Addition vs. v3 | What it buys |
|:---|:---|
| **Second dataset: CheXpert-v1.0-small** (Kaggle `ashery/chexpert`, Stanford Hospital, genuinely independent from NIH) | Tests whether H1/H2/H3a generalize beyond one institution. The full pipeline now runs on both datasets. |
| **H4: Leakage-direction ablation** (iterative nullspace projection) | Converts H3b from correlational to interventional evidence — removes the age-decoding direction from embeddings and re-measures whether the calibration gap shrinks, rather than just observing the two quantities co-vary across models. |
| **Leave-one-model-out jackknife on H3b** | Tests whether the rho=1.0 cross-model correlation is robust to which 5 models were chosen, not just a property of this exact set. |
| **Continuous/quartile age dose-response** | Replaces the binary median-split age with a quartile analysis — addresses the "arbitrary threshold" objection by showing whether the calibration gap scales smoothly with age rather than appearing only at one cutoff. |
| **Multi-seed stability check** on the ResNet-50 raw-vs-matched 60x gap swing | Confirms (or disconfirms) that the prevalence-matching procedure's largest single effect is stable across resampling seeds, not an artifact of which patients got dropped in matching. |
| **Two new figures**: cross-dataset replication comparison, replication-consistency scatter | Lets a reviewer see directly whether the NIH findings hold on Stanford data, rather than taking it on faith from text. |

> **Data**: NIH ChestX-ray14 (Kaggle `nih-chest-xrays/data`) **and** CheXpert-v1.0-small (Kaggle `ashery/chexpert`) — attach both via "+ Add Input". The notebook degrades gracefully if only one is attached (CheXpert-dependent cells are skipped with a clear message, not a crash).
> **Hardware**: T4 GPU, multi-core CPU · **Runtime**: ~3.5–5 hours with both datasets at full N_SAMPLES (embedding extraction and permutation tests now run twice) · **Output**: `outputs.zip` auto-downloaded.
> **Honest framing reminder**: H1, H2, H3a are the primary, FDR-corrected claims. H3b, H4, the jackknife, and the dose-response curve are robustness/mechanism analyses — real evidence, but each explicitly scoped as exploratory or secondary in the printed output, not silently elevated to primary-claim status.


In [ ]:
# =============================================================================
# CELL 1 — Install & Setup
# =============================================================================
import subprocess, sys, gc, json, random, shutil, warnings, os
from pathlib import Path

def _pip(*pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=True)

_pip("open_clip_torch>=2.24.0", "kagglehub>=0.3.0", "joblib>=1.3",
     "scikit-learn>=1.4", "scipy>=1.13", "pandas>=2.2", "timm>=1.0")

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib
import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image
from scipy.stats import spearmanr
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.decomposition import PCA
import torchvision.transforms as T
from torchvision.models import resnet50, ResNet50_Weights
from joblib import Parallel, delayed
import open_clip

warnings.filterwarnings("ignore")

# ── Reproducibility ─────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark     = False

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device  : {DEVICE}")
if DEVICE.type == "cuda":
    props = torch.cuda.get_device_properties(0)
    print(f"GPU     : {props.name}  |  VRAM: {props.total_memory/1e9:.1f} GB")
print(f"PyTorch : {torch.__version__}  |  OpenCLIP: {open_clip.__version__}")

# ── Output dirs ──────────────────────────────────────────────────────────────
OUTDIR = Path("outputs")
for d in ("figures", "tables", "logs", "embeddings"):
    (OUTDIR/d).mkdir(parents=True, exist_ok=True)

# ── ECCV figure style (Wong 2011 colorblind-safe palette) ───────────────────
FIG_W_SINGLE = 4.80   # 122 mm
FIG_W_DOUBLE = 7.16   # 182 mm
DPI          = 300
PALETTE = ["#E69F00","#56B4E9","#009E73","#D55E00","#0072B2","#CC79A7","#F0E442"]

matplotlib.rcParams.update({
    "font.family": "serif", "font.size": 8,
    "axes.labelsize": 8, "axes.titlesize": 9,
    "xtick.labelsize": 7, "ytick.labelsize": 7,
    "legend.fontsize": 7, "figure.dpi": 120,
    "savefig.dpi": DPI, "savefig.bbox": "tight",
    "savefig.pad_inches": 0.02,
    "lines.linewidth": 1.3, "axes.linewidth": 0.7,
})

# ── Statistical config ────────────────────────────────────────────────────────
N_FOLDS       = 5
N_PERM        = 150     # permutation test resamples (full CV retrain each time)
N_BOOT        = 1500    # bootstrap resamples (fast: resamples fixed oof predictions)
ALPHA         = 0.05    # significance threshold BEFORE multiple-comparison correction
FDR_Q         = 0.05    # Benjamini-Hochberg false-discovery-rate threshold
N_JOBS        = max(1, (os.cpu_count() or 4) - 1)   # leave one core free

# ── v4 robustness-suite config ───────────────────────────────────────────────
N_INLP_ITERS   = 8      # max directions removed per iterative-nullspace-projection ablation
INLP_AUROC_FLOOR = 0.55 # stop ablating once residual leakage AUROC is this close to chance
N_MATCH_SEEDS  = 10     # resampling seeds for the multi-seed prevalence-matching stability check
N_AGE_QUANTILES = 4     # bins for the continuous age dose-response analysis

def _despine(ax):
    ax.spines[["top","right"]].set_visible(False)

def _savefig(fig, name, double=False):
    w = FIG_W_DOUBLE if double else FIG_W_SINGLE
    fig.set_size_inches(w, fig.get_size_inches()[1])
    for ext in ("pdf","png"):
        fig.savefig(OUTDIR/"figures"/f"{name}.{ext}", dpi=DPI,
                   bbox_inches="tight", pad_inches=0.02)

print(f"Setup complete.  N_FOLDS={N_FOLDS}  N_PERM={N_PERM}  N_BOOT={N_BOOT}  "
      f"FDR_Q={FDR_Q}  N_JOBS={N_JOBS}")
print(f"v4 robustness config: N_INLP_ITERS={N_INLP_ITERS}  N_MATCH_SEEDS={N_MATCH_SEEDS}  "
      f"N_AGE_QUANTILES={N_AGE_QUANTILES}")


## Section 2 — Problem Definition

Let $f_\theta$ be a **frozen** medical or natural-image foundation model, and $\mathbf{Z} = f_\theta(\mathcal{X}) \in \mathbb{R}^{n \times d}$ the embedding matrix of $n$ chest X-rays. Let $y_{\text{sex}}, y_{\text{age}}, y_{\text{dis}} \in \{0,1\}^n$ be patient sex, median-split age, and any-pathology-present labels — **none of which were used during $f_\theta$'s pretraining.**

### Primary Hypotheses (high statistical power, FDR-corrected, tested on both datasets)

| | Claim | Test |
|:--|:---|:---|
| **H1** | A linear probe on frozen $\mathbf{Z}$ predicts $y_{\text{sex}}$ and $y_{\text{age}}$ significantly above chance | 5-fold CV out-of-fold AUROC + permutation test, FDR-corrected across 15 tests |
| **H2** | Medical-pretrained embeddings (BioMedCLIP) leak **more** demographic signal than natural-pretrained embeddings on **identical images** | Paired bootstrap, FDR-corrected across the 8 sex+age comparisons |
| **H3a** | The sex/age calibration gap persists after prevalence-matching | Permutation test on matched-gap, FDR-corrected across 10 tests |
| **H5** | H1, H2, and H3a replicate on an independent dataset (CheXpert, Stanford Hospital — different institution, scanner fleet, and labeling pipeline than NIH) | Full pipeline re-run on CheXpert; cross-dataset consistency reported directly, not asserted |

### Mechanism & Robustness Analyses (explicitly secondary — real evidence, not silently elevated)

| | Claim | Test |
|:--|:---|:---|
| **H3b** | Per-model leakage AUROC correlates with that model's calibration gap (the axis with real H3a signal: age) | Spearman $\rho$ across 5 points, each checked for individual FDR-significance before being trusted |
| **H4** | Removing the age-decoding direction from embeddings (iterative nullspace projection) reduces the downstream calibration gap | Re-run the full disease-probe → ECE → prevalence-match pipeline on ablated embeddings; compare gap before/after, with a sanity-check confirming residual leakage AUROC actually drops near chance |
| **Jackknife** | The H3b correlation isn't fragile to which 5 models were used | Leave-one-model-out recomputation of $\rho$, 5 times |
| **Dose-response** | The age-calibration gap scales with age, not just with a single arbitrary median-split threshold | ECE computed per age quartile rather than per binary group |
| **Multi-seed stability** | The ResNet-50 age-axis raw-to-matched gap swing (a 60x jump in earlier runs) is a stable property of the model, not an artifact of which patients got dropped during prevalence-matching | Re-run matching with 10 different random seeds, report the distribution |

### Why H4 matters more than it might look

H3b shows leakage magnitude and calibration gap *covary* across 5 models — but that's still observational: something else could drive both. H4 is an intervention: surgically remove the specific direction in embedding space that encodes age, then ask whether the calibration gap responds. If it shrinks, that's evidence the leakage geometry is mechanistically implicated, not just correlated with the outcome.

### Calibration metric

$$
\text{ECE} = \sum_{m=1}^{M} \frac{|B_m|}{n} \left| \text{acc}(B_m) - \text{conf}(B_m) \right|, \qquad M = 10 \text{ equal-width bins}
$$


In [ ]:
# =============================================================================
# CELL 3 — Core Statistical Method
# All claims are backed by out-of-fold CV, permutation tests, and bootstrap CIs.
# =============================================================================

def probe_cv_oof(Z: np.ndarray, y: np.ndarray, n_folds: int = N_FOLDS,
                 seed: int = SEED) -> np.ndarray:
    """5-fold stratified CV logistic-regression probe.
    Returns out-of-fold predicted P(y=1) for every sample (unbiased)."""
    oof = np.zeros(len(y), dtype=np.float64)
    skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=seed)
    for tr_idx, te_idx in skf.split(Z, y):
        clf = LogisticRegression(max_iter=2000, class_weight="balanced",
                                 random_state=seed, solver="lbfgs", C=1.0)
        clf.fit(Z[tr_idx], y[tr_idx])
        oof[te_idx] = clf.predict_proba(Z[te_idx])[:, 1]
    return oof

def _single_permutation_auroc(Z: np.ndarray, y: np.ndarray, perm_seed: int) -> float:
    """One permutation: shuffle y, retrain the full 5-fold CV probe, return null AUROC.
    Factored out as a standalone function so it can run in parallel workers."""
    rng    = np.random.default_rng(perm_seed)
    y_perm = rng.permutation(y)
    oof_perm = probe_cv_oof(Z, y_perm, seed=perm_seed)
    return float(roc_auc_score(y_perm, oof_perm))

def permutation_test_auroc(Z: np.ndarray, y: np.ndarray, observed_auroc: float,
                           n_perm: int = N_PERM, seed: int = SEED,
                           n_jobs: int = N_JOBS) -> float:
    """Full retrain-per-permutation test (gold standard, not just label-shuffle-on-fixed-preds).
    Parallelized across CPU cores -- logistic regression's BLAS calls release the GIL,
    so thread-based parallelism gives a real speedup without the pickling overhead of
    process-based parallelism (Z can be large: up to ~2048-dim x 15000 rows).
    Returns p-value = P(null AUROC >= observed)."""
    null = Parallel(n_jobs=n_jobs, prefer="threads")(
        delayed(_single_permutation_auroc)(Z, y, seed + i + 1) for i in range(n_perm)
    )
    null = np.array(null)
    return float((1 + np.sum(null >= observed_auroc)) / (1 + n_perm))

def benjamini_hochberg(pvals, q: float = FDR_Q) -> np.ndarray:
    """Benjamini-Hochberg FDR correction across a family of m hypothesis tests.
    Returns a boolean array (same order as input) marking which tests remain
    significant after controlling the expected false-discovery rate at q."""
    pvals = np.asarray(pvals, dtype=float)
    m     = len(pvals)
    order = np.argsort(pvals)
    ranked = pvals[order]
    thresh = (np.arange(1, m + 1) / m) * q
    below  = ranked <= thresh
    sig    = np.zeros(m, dtype=bool)
    if below.any():
        max_idx = int(np.max(np.where(below)[0]))
        sig[order[:max_idx + 1]] = True
    return sig

def bootstrap_ci_auroc(oof_probs: np.ndarray, y: np.ndarray,
                       n_boot: int = N_BOOT, seed: int = SEED) -> tuple:
    """Bootstrap 95% CI on AUROC via resampling FIXED oof predictions (fast, no refit)."""
    rng = np.random.default_rng(seed)
    n   = len(y)
    boots = []
    for _ in range(n_boot):
        idx = rng.choice(n, n, replace=True)
        if len(np.unique(y[idx])) < 2:
            continue
        boots.append(roc_auc_score(y[idx], oof_probs[idx]))
    boots = np.array(boots)
    return float(np.percentile(boots, 2.5)), float(np.percentile(boots, 97.5)), float(boots.std())

def paired_bootstrap_diff(oof_a: np.ndarray, oof_b: np.ndarray, y: np.ndarray,
                          n_boot: int = N_BOOT, seed: int = SEED) -> dict:
    """Paired bootstrap on AUROC_a - AUROC_b using the SAME resampled indices for both
    (removes shared sampling variance -> correct CI on the difference, used for H2)."""
    rng = np.random.default_rng(seed)
    n   = len(y)
    diffs = []
    for _ in range(n_boot):
        idx = rng.choice(n, n, replace=True)
        if len(np.unique(y[idx])) < 2:
            continue
        diffs.append(roc_auc_score(y[idx], oof_a[idx]) - roc_auc_score(y[idx], oof_b[idx]))
    diffs = np.array(diffs)
    p_two_sided = float(2 * min((diffs <= 0).mean(), (diffs >= 0).mean()))
    return dict(
        mean=float(diffs.mean()),
        ci_lo=float(np.percentile(diffs, 2.5)),
        ci_hi=float(np.percentile(diffs, 97.5)),
        p=min(p_two_sided, 1.0),
    )

def compute_ece(probs: np.ndarray, labels: np.ndarray, n_bins: int = 10) -> float:
    """Standard expected calibration error, equal-width bins on predicted probability."""
    bins = np.linspace(0.0, 1.0, n_bins + 1)
    n    = len(labels)
    ece  = 0.0
    for lo, hi in zip(bins[:-1], bins[1:]):
        mask = (probs >= lo) & (probs < hi) if hi < 1.0 else (probs >= lo) & (probs <= hi)
        if mask.sum() == 0:
            continue
        acc  = labels[mask].mean()
        conf = probs[mask].mean()
        ece += (mask.sum() / n) * abs(acc - conf)
    return float(ece)

def prevalence_matched_ece_gap(probs: np.ndarray, labels: np.ndarray,
                               group: np.ndarray, seed: int = SEED) -> dict:
    """Downsample the higher-prevalence demographic group's POSITIVE cases until
    prevalence matches the lower-prevalence group -> any residual ECE gap cannot
    be explained by differential disease prevalence."""
    rng = np.random.default_rng(seed)
    gA, gB = np.unique(group)
    idx_A, idx_B = np.where(group == gA)[0], np.where(group == gB)[0]

    prev_A = labels[idx_A].mean()
    prev_B = labels[idx_B].mean()
    raw_gap = abs(compute_ece(probs[idx_A], labels[idx_A]) -
                  compute_ece(probs[idx_B], labels[idx_B]))

    target = min(prev_A, prev_B)
    def _match(idx, prev):
        """Downsample positives so that pos/(pos+neg) == target, holding neg fixed."""
        pos, neg = idx[labels[idx] == 1], idx[labels[idx] == 0]
        if prev <= target + 1e-9:
            return idx
        n_keep_pos = max(1, min(len(pos), int(round(target / (1 - target) * len(neg)))))
        keep_pos   = rng.choice(pos, n_keep_pos, replace=False) if n_keep_pos < len(pos) else pos
        return np.sort(np.concatenate([keep_pos, neg]))

    idx_A_m = _match(idx_A, prev_A)
    idx_B_m = _match(idx_B, prev_B)
    matched_gap = abs(compute_ece(probs[idx_A_m], labels[idx_A_m]) -
                      compute_ece(probs[idx_B_m], labels[idx_B_m]))

    return dict(raw_gap=raw_gap, matched_gap=matched_gap,
                prev_A=float(prev_A), prev_B=float(prev_B),
                n_A_matched=len(idx_A_m), n_B_matched=len(idx_B_m),
                idx_A_matched=idx_A_m, idx_B_matched=idx_B_m)

def permutation_test_ece_gap(probs: np.ndarray, labels: np.ndarray, group: np.ndarray,
                             idx_A: np.ndarray, idx_B: np.ndarray, observed_gap: float,
                             n_perm: int = N_PERM, seed: int = SEED) -> float:
    """Permute group identity WITHIN the prevalence-matched subsample; test whether
    observed gap exceeds what random group assignment would produce."""
    rng  = np.random.default_rng(seed)
    pool = np.concatenate([idx_A, idx_B])
    nA   = len(idx_A)
    null = np.empty(n_perm)
    for i in range(n_perm):
        perm = rng.permutation(pool)
        pa, pb = perm[:nA], perm[nA:]
        null[i] = abs(compute_ece(probs[pa], labels[pa]) - compute_ece(probs[pb], labels[pb]))
    return float((1 + np.sum(null >= observed_gap)) / (1 + n_perm))

def iterative_nullspace_projection(Z: np.ndarray, y: np.ndarray,
                                   max_iters: int = N_INLP_ITERS,
                                   auroc_floor: float = INLP_AUROC_FLOOR,
                                   seed: int = SEED) -> dict:
    """H4: INLP-style ablation (Ravfogel et al. 2020). Iteratively identifies the
    linear direction most predictive of y, projects it out of Z, and repeats until
    the residual leakage AUROC drops near chance or max_iters is reached. This is
    what converts H3b from correlational to interventional: if downstream
    calibration improves after this ablation, the leakage direction is implicated
    mechanistically, not just associated across models."""
    Z_cur = Z.copy()
    auroc_trace = []
    directions = []
    for it in range(max_iters):
        oof   = probe_cv_oof(Z_cur, y, seed=seed)
        auroc = roc_auc_score(y, oof)
        auroc_trace.append(float(auroc))
        if auroc <= auroc_floor:
            break
        clf = LogisticRegression(max_iter=2000, C=1.0)
        clf.fit(Z_cur, y)
        w     = clf.coef_.ravel()
        w_hat = w / (np.linalg.norm(w) + 1e-12)
        directions.append(w_hat)
        Z_cur = Z_cur - np.outer(Z_cur @ w_hat, w_hat)
    return dict(Z_ablated=Z_cur, auroc_trace=auroc_trace,
                n_directions_removed=len(directions), directions=directions)

def jackknife_correlation(leakage_aurocs, calib_gaps, model_names) -> list:
    """Leave-one-model-out recomputation of the H3b Spearman correlation -- tests
    whether rho is fragile to which specific 5 models happened to be included."""
    n = len(model_names)
    results = []
    for i in range(n):
        mask = np.arange(n) != i
        if mask.sum() >= 3:
            rho, p = spearmanr(np.array(leakage_aurocs)[mask], np.array(calib_gaps)[mask])
        else:
            rho, p = float("nan"), float("nan")
        results.append(dict(excluded_model=model_names[i], rho=float(rho), p=float(p),
                            n_remaining=int(mask.sum())))
    return results

def multiseed_matched_gap(probs: np.ndarray, labels: np.ndarray, group: np.ndarray,
                          n_seeds: int = N_MATCH_SEEDS, base_seed: int = SEED) -> dict:
    """Re-runs prevalence-matching with multiple random seeds for the downsampling
    step, to check whether a given (model, axis) matched-gap estimate is stable or
    sensitive to exactly which patients got dropped during matching."""
    gaps = [prevalence_matched_ece_gap(probs, labels, group, seed=base_seed + i)["matched_gap"]
           for i in range(n_seeds)]
    gaps = np.array(gaps)
    return dict(mean=float(gaps.mean()), std=float(gaps.std()),
               min=float(gaps.min()), max=float(gaps.max()), all_gaps=gaps.tolist())

def quartile_age_dose_response(probs: np.ndarray, labels: np.ndarray,
                               age_continuous: np.ndarray,
                               n_quantiles: int = N_AGE_QUANTILES) -> list:
    """Bins age into quantiles instead of a single median split, to test whether
    the calibration gap scales smoothly with age (a dose-response pattern) or only
    appears because of where the binary threshold happens to fall."""
    edges = np.quantile(age_continuous, np.linspace(0, 1, n_quantiles + 1))
    edges = edges.copy(); edges[0] -= 1e-6
    bin_idx = np.digitize(age_continuous, edges[1:-1], right=True)

    bin_results = []
    for q in range(n_quantiles):
        mask = bin_idx == q
        if mask.sum() < 10:
            continue
        bin_results.append(dict(
            quantile=q, n=int(mask.sum()),
            age_lo=float(edges[q]), age_hi=float(edges[q+1]),
            age_mean=float(age_continuous[mask].mean()),
            ece=float(compute_ece(probs[mask], labels[mask])),
            prevalence=float(labels[mask].mean())))
    if bin_results:
        base = bin_results[0]["ece"]
        for r in bin_results:
            r["gap_vs_lowest_quantile"] = r["ece"] - base
    return bin_results

print("Statistical method functions defined.")
print("  H1/H2 : probe_cv_oof -> bootstrap_ci_auroc / paired_bootstrap_diff / permutation_test_auroc")
print("  H3a   : prevalence_matched_ece_gap -> permutation_test_ece_gap")
print("  All   : benjamini_hochberg(pvals, q) -- FDR correction applied to every multi-test family")
print("  v4    : iterative_nullspace_projection (H4) / jackknife_correlation / "
      "multiseed_matched_gap / quartile_age_dose_response")


In [ ]:
# =============================================================================
# CELL 4 — Baselines
#   (1) Raw-pixel PCA probe  -> tests whether leakage exceeds what's trivially
#       visible in pixel intensities (bone density, body habitus, etc.)
#   (2) Chance baseline       -> AUROC = 0.5
# =============================================================================

def extract_pixel_pca_features(images_u8: np.ndarray, n_components: int = 64,
                               seed: int = SEED) -> np.ndarray:
    """Flatten images, standardize, PCA-reduce. A baseline that uses ONLY raw
    pixel intensity/texture statistics -- no learned semantic representation."""
    flat = images_u8.reshape(len(images_u8), -1).astype(np.float32) / 255.0
    flat = (flat - flat.mean(axis=0, keepdims=True)) / (flat.std(axis=0, keepdims=True) + 1e-6)
    k    = min(n_components, flat.shape[0] - 1, flat.shape[1])
    pca  = PCA(n_components=k, random_state=seed)
    return pca.fit_transform(flat).astype(np.float32)

CHANCE_AUROC = 0.500
print("Baselines defined: Pixel-PCA probe | Chance (AUROC=0.5).")


In [ ]:
# =============================================================================
# CELL 5 — Datasets: NIH ChestX-ray14 (required) + CheXpert-v1.0-small (optional)
#
# Both are Kaggle-native -- attach via "+ Add Input", no registration needed for
# either (CheXpert-v1.0-small is a third-party Kaggle mirror of Stanford's
# original CheXpert release, which normally requires a Stanford AIMI data-use
# agreement; the Kaggle mirror sidesteps that). If only NIH is attached, the
# notebook still runs correctly -- CheXpert-dependent cells are skipped with a
# clear message rather than crashing.
#
# IMPORTANT SCHEMA DIFFERENCE: NIH's "Image Index" filenames are globally
# unique, so a flat filename->path index works. CheXpert's filenames are NOT
# unique (every patient folder contains a "view1_frontal.jpg"), so CheXpert
# needs path-based resolution, not filename-based indexing.
# =============================================================================
IMG_SIZE   = 224
N_SAMPLES  = 15000

KAGGLE_INPUT = Path("/kaggle/input")
assert KAGGLE_INPUT.exists(), "This doesn't look like a Kaggle Notebook environment."

candidate_dirs = [p for p in KAGGLE_INPUT.iterdir() if p.is_dir()]
print("Datasets currently attached to this notebook:")
for d in candidate_dirs:
    print(" -", d)
if not candidate_dirs:
    raise RuntimeError("No dataset attached. Click '+ Add Input' and attach at least NIH Chest X-rays.")
SEARCH_ROOT = candidate_dirs[0]

def _load_image(path, size=IMG_SIZE):
    img = Image.open(path).convert("L").resize((size, size), Image.BILINEAR)
    return np.array(img, dtype=np.uint8)

def _stratified_subsample(df, dis_col, n_target, seed=SEED):
    if len(df) <= n_target:
        return df
    rng = np.random.default_rng(seed)
    pos_df, neg_df = df[df[dis_col] == 1], df[df[dis_col] == 0]
    frac  = n_target / len(df)
    pos_n = int(round(len(pos_df) * frac))
    neg_n = n_target - pos_n
    pos_idx = rng.choice(pos_df.index, min(pos_n, len(pos_df)), replace=False)
    neg_idx = rng.choice(neg_df.index, min(neg_n, len(neg_df)), replace=False)
    return df.loc[np.concatenate([pos_idx, neg_idx])].sample(frac=1.0, random_state=seed)

# ── NIH ChestX-ray14 loader (multi-folder, filename-indexed) ─────────────────
def load_nih_dataset(search_root, n_target=N_SAMPLES):
    csv_candidates = list(search_root.rglob("*labels*.csv")) + list(search_root.rglob("Data_Entry*.csv"))
    if not csv_candidates:
        return None
    labels_csv = max(csv_candidates, key=lambda p: p.stat().st_size)
    print(f"[NIH] Labels CSV: {labels_csv}")

    img_dirs = [p for p in search_root.rglob("*") if p.is_dir() and
               any(f.suffix.lower() == ".png" for f in list(p.iterdir())[:5])]
    if not img_dirs:
        return None
    print(f"[NIH] Found {len(img_dirs)} image director{'y' if len(img_dirs)==1 else 'ies'}")
    image_index = {}
    for d in img_dirs:
        for f in d.glob("*.png"):
            image_index[f.name] = f
    print(f"[NIH] Total unique images indexed: {len(image_index)}")

    df = pd.read_csv(labels_csv)
    df.columns = [c.strip() for c in df.columns]

    def _find_col(cands):
        for c in cands:
            if c in df.columns: return c
        raise KeyError(f"None of {cands} in {list(df.columns)}")

    col_img  = _find_col(["Image Index", "Image_Index", "image"])
    col_age  = _find_col(["Patient Age", "Patient_Age", "age"])
    col_sex  = _find_col(["Patient Gender", "Patient_Gender", "gender", "sex"])
    col_find = _find_col(["Finding Labels", "Finding_Labels", "labels"])
    col_pid  = "Patient ID" if "Patient ID" in df.columns else None

    def _parse_age(v):
        s = str(v).strip()
        digits = "".join(ch for ch in s if ch.isdigit())
        return int(digits) if digits else np.nan

    df["age_num"] = df[col_age].apply(_parse_age)
    df = df[(df["age_num"] > 0) & (df["age_num"] <= 100)]
    df["sex_bin"] = (df[col_sex].astype(str).str.upper().str[0] == "M").astype(int)
    df["dis_bin"] = (~df[col_find].astype(str).str.contains("No Finding", case=False)).astype(int)
    df["img_path"] = df[col_img].map(image_index)
    df = df[df["img_path"].notna()]
    if df.empty:
        return None

    if col_pid is not None:
        n0 = len(df)
        df = df.sort_values("age_num").drop_duplicates(subset=col_pid, keep="first")
        print(f"[NIH] Deduplicated by Patient ID: {n0} -> {len(df)} rows")

    age_median = float(df["age_num"].median())
    df["age_bin"] = (df["age_num"] >= age_median).astype(int)
    df = _stratified_subsample(df, "dis_bin", n_target).reset_index(drop=True)
    print(f"[NIH] Final sample size: {len(df)}")

    images = np.stack([_load_image(p) for p in df["img_path"]])[:, np.newaxis, :, :]
    return dict(name="NIH", images=images, sex_bin=df["sex_bin"].to_numpy(),
               age_bin=df["age_bin"].to_numpy(), age_continuous=df["age_num"].to_numpy(float),
               dis_bin=df["dis_bin"].to_numpy(), age_median=age_median, n_total=len(df))

# ── CheXpert-v1.0-small loader (path-based, NOT filename-indexed) ────────────
def load_chexpert_dataset(search_root, n_target=N_SAMPLES):
    csv_candidates = list(search_root.rglob("train.csv"))
    if not csv_candidates:
        return None
    train_csv = csv_candidates[0]
    print(f"[CheXpert] Labels CSV: {train_csv}")

    train_dirs = [d for d in search_root.rglob("train") if d.is_dir()]
    if not train_dirs:
        print("[CheXpert] Could not locate a 'train' image folder -- skipping CheXpert.")
        return None
    chexpert_root = train_dirs[0].parent
    print(f"[CheXpert] Image root resolved to: {chexpert_root}")

    df = pd.read_csv(train_csv)
    df.columns = [c.strip() for c in df.columns]
    required = {"Path", "Sex", "Age", "No Finding"}
    if not required.issubset(df.columns):
        print(f"[CheXpert] CSV missing expected columns {required - set(df.columns)} -- skipping.")
        return None

    # Resolve the path-prefix-strip count using ONE sample row, then apply to all
    sample_rel = df["Path"].iloc[0]
    parts = Path(sample_rel).parts
    strip_n = None
    for i in range(len(parts)):
        if (chexpert_root / Path(*parts[i:])).exists():
            strip_n = i
            break
    if strip_n is None:
        print(f"[CheXpert] Could not resolve image path structure for sample '{sample_rel}' -- skipping.")
        return None
    print(f"[CheXpert] Path resolution: stripping {strip_n} leading component(s) from CSV paths")

    # Keep frontal views only -- NIH is entirely frontal (PA/AP); CheXpert mixes
    # frontal and lateral, and mixing view types would confound the comparison.
    if "Frontal/Lateral" in df.columns:
        n0 = len(df)
        df = df[df["Frontal/Lateral"] == "Frontal"]
        print(f"[CheXpert] Frontal-only filter: {n0} -> {len(df)} rows")

    df = df[df["Sex"].isin(["Male", "Female"])].copy()
    df["sex_bin"] = (df["Sex"] == "Male").astype(int)
    df = df[(df["Age"] > 0) & (df["Age"] <= 100)]
    df["dis_bin"] = (df["No Finding"] != 1.0).astype(int)
    df["img_path"] = df["Path"].apply(lambda p: chexpert_root / Path(*Path(p).parts[strip_n:]))
    exists_mask = df["img_path"].apply(lambda p: p.exists())
    df = df[exists_mask]
    if df.empty:
        print("[CheXpert] No resolved image paths actually exist on disk -- skipping.")
        return None
    print(f"[CheXpert] Rows with resolvable images: {len(df)}")

    def _patient_id(rel_path):
        for part in Path(rel_path).parts:
            if part.startswith("patient"):
                return part
        return rel_path
    df["patient_id"] = df["Path"].apply(_patient_id)
    n0 = len(df)
    df = df.sort_values("Age").drop_duplicates(subset="patient_id", keep="first")
    print(f"[CheXpert] Deduplicated by patient: {n0} -> {len(df)} rows")

    age_median = float(df["Age"].median())
    df["age_bin"] = (df["Age"] >= age_median).astype(int)
    df = _stratified_subsample(df, "dis_bin", n_target).reset_index(drop=True)
    print(f"[CheXpert] Final sample size: {len(df)}")

    images = np.stack([_load_image(p) for p in df["img_path"]])[:, np.newaxis, :, :]
    return dict(name="CheXpert", images=images, sex_bin=df["sex_bin"].to_numpy(),
               age_bin=df["age_bin"].to_numpy(), age_continuous=df["Age"].to_numpy(float),
               dis_bin=df["dis_bin"].to_numpy(), age_median=age_median, n_total=len(df))

# ── Load both, build the DATASETS registry ────────────────────────────────────
print("\n" + "="*62 + "\nLoading NIH ChestX-ray14...\n" + "="*62)
_nih = load_nih_dataset(SEARCH_ROOT, N_SAMPLES)
assert _nih is not None, "NIH dataset is REQUIRED and could not be loaded. Check that it's attached."

print("\n" + "="*62 + "\nLoading CheXpert-v1.0-small (optional)...\n" + "="*62)
_chexpert = load_chexpert_dataset(SEARCH_ROOT, N_SAMPLES)
CHEXPERT_AVAILABLE = _chexpert is not None
if not CHEXPERT_AVAILABLE:
    print("\nCheXpert NOT available -- the notebook will run NIH-only. To add it: "
          "'+ Add Input' -> search 'chexpert' -> attach 'CheXpert-v1.0-small' (Kaggle "
          "user 'ashery') -> re-run from this cell.")

DATASETS = {"NIH": _nih}
if CHEXPERT_AVAILABLE:
    DATASETS["CheXpert"] = _chexpert
DATASET_NAMES = list(DATASETS.keys())

# Backward-compatible flat variables for the PRIMARY dataset (NIH) -- existing
# H1/H2/H3 statistical cells and Figures 1-6 use these unchanged.
ALL_IMAGES = DATASETS["NIH"]["images"]
Y_SEX      = DATASETS["NIH"]["sex_bin"]
Y_AGE      = DATASETS["NIH"]["age_bin"]
Y_AGE_CONT = DATASETS["NIH"]["age_continuous"]
Y_DIS      = DATASETS["NIH"]["dis_bin"]
N_TOTAL    = DATASETS["NIH"]["n_total"]

for ds_name, ds in DATASETS.items():
    if ds["n_total"] < 0.3 * N_SAMPLES:
        print(f"\n{'!'*70}\n! WARNING: {ds_name} only has {ds['n_total']:,} images "
              f"(target was {N_SAMPLES:,}). Statistical power for this dataset's\n"
              f"! tests will be reduced accordingly.\n{'!'*70}")
    print(f"\n[{ds_name}] N={ds['n_total']}  sex(M)={ds['sex_bin'].mean():.3f}  "
          f"age(old)={ds['age_bin'].mean():.3f}  disease={ds['dis_bin'].mean():.3f}")


In [ ]:
# =============================================================================
# CELL 6 — Figure 0: Sample CXRs with Demographic Labels (sanity check)
# One row per attached dataset.
# =============================================================================
fig, axes = plt.subplots(len(DATASET_NAMES), 5, figsize=(FIG_W_DOUBLE, 1.7*len(DATASET_NAMES)))
if len(DATASET_NAMES) == 1:
    axes = axes[np.newaxis, :]

for row, ds_name in enumerate(DATASET_NAMES):
    ds = DATASETS[ds_name]
    rng_show  = np.random.default_rng(SEED)
    show_idx  = rng_show.choice(ds["n_total"], 5, replace=False)
    for col, idx in enumerate(show_idx):
        ax = axes[row, col]
        ax.imshow(ds["images"][idx, 0], cmap="gray")
        sex_lbl = "M" if ds["sex_bin"][idx] == 1 else "F"
        age_lbl = "old" if ds["age_bin"][idx] == 1 else "young"
        dis_lbl = "+" if ds["dis_bin"][idx] == 1 else "-"
        ax.text(0.5, -0.06, f"{sex_lbl} / {age_lbl} / dis{dis_lbl}", transform=ax.transAxes,
                fontsize=6.5, ha="center", va="top")
        ax.axis("off")
        if col == 0:
            ax.text(-0.12, 0.5, ds_name, transform=ax.transAxes, fontsize=8, fontweight="bold",
                    ha="right", va="center", rotation=90)

fig.tight_layout(pad=0.6)
_savefig(fig, "fig0_sample_cxrs", double=True)
plt.show()
plt.close(fig)
print("Figure 0 saved.")
print("Caption (for paper): Sample CXRs with sex / age-group / disease-status labels, "
      "one row per dataset.")


In [ ]:
# =============================================================================
# CELL 7 — Embedding Extraction: 5 Models Spanning 4 Pretraining Paradigms
#   BioMedCLIP    : medical-domain, text-supervised (contrastive)
#   CLIP B/16,B32 : natural-domain, text-supervised (contrastive)
#   DINOv2-S      : natural-domain, self-supervised
#   ResNet-50     : natural-domain, fully-supervised (ImageNet)
#
# Each model is loaded ONCE, then run on every attached dataset's images before
# being unloaded -- avoids redundant model-loading overhead across datasets.
# =============================================================================
from tqdm.auto import tqdm

BATCH_SIZE = 64

def _to_pil_rgb(imgs_np: np.ndarray) -> list:
    out = []
    for img in imgs_np:
        if img.shape[0] == 1:
            img = np.repeat(img, 3, axis=0)
        out.append(Image.fromarray(img.transpose(1, 2, 0)))
    return out

IMAGENET_TRANSFORM = T.Compose([
    T.Resize(256), T.CenterCrop(224), T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

@torch.no_grad()
def extract_openclip(model, preprocess, imgs_np, desc=""):
    model.eval()
    pils, feats = _to_pil_rgb(imgs_np), []
    for i in tqdm(range(0, len(pils), BATCH_SIZE), desc=desc, leave=False):
        batch = torch.stack([preprocess(p) for p in pils[i:i+BATCH_SIZE]]).to(DEVICE)
        f = F.normalize(model.encode_image(batch).float(), dim=-1)
        feats.append(f.cpu().numpy())
    return np.concatenate(feats, axis=0)

@torch.no_grad()
def extract_torchvision(model, imgs_np, desc=""):
    model.eval()
    pils, feats = _to_pil_rgb(imgs_np), []
    for i in tqdm(range(0, len(pils), BATCH_SIZE), desc=desc, leave=False):
        batch = torch.stack([IMAGENET_TRANSFORM(p) for p in pils[i:i+BATCH_SIZE]]).to(DEVICE)
        f = F.normalize(model(batch).float(), dim=-1)
        feats.append(f.cpu().numpy())
    return np.concatenate(feats, axis=0)

# ── Model loaders (each wrapped in try/except — missing model is SKIPPED, not faked) ──
def _load_biomedclip():
    m, _, p = open_clip.create_model_and_transforms(
        "hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224")
    return m.to(DEVICE).eval(), p, "openclip"

def _load_clip(variant, pretrained="openai"):
    m, _, p = open_clip.create_model_and_transforms(variant, pretrained=pretrained)
    return m.to(DEVICE).eval(), p, "openclip"

def _load_dinov2():
    m = torch.hub.load("facebookresearch/dinov2", "dinov2_vits14")
    return m.to(DEVICE).eval(), None, "torchvision"

def _load_resnet50():
    m = resnet50(weights=ResNet50_Weights.IMAGENET1K_V2)
    m.fc = nn.Identity()
    return m.to(DEVICE).eval(), None, "torchvision"

MODEL_LOADERS = [
    ("BioMedCLIP",     _load_biomedclip),
    ("CLIP ViT-B/16",  lambda: _load_clip("ViT-B-16")),
    ("CLIP ViT-B/32",  lambda: _load_clip("ViT-B-32")),
    ("DINOv2-S/14",    _load_dinov2),
    ("ResNet-50",      _load_resnet50),
]

EMBEDDINGS_BY_DS = {ds: {} for ds in DATASET_NAMES}     # {dataset: {model_name: Z}}
MODEL_NAMES = []

for mname, loader_fn in MODEL_LOADERS:
    print(f"\n{'='*55}\n  {mname}\n{'='*55}")
    try:
        model, preprocess, kind = loader_fn()
    except Exception as exc:
        print(f"  SKIPPED ({mname}): {exc}")
        continue

    n_p = sum(p.numel() for p in model.parameters()) / 1e6
    print(f"  Params: {n_p:.1f}M")

    for ds_name in DATASET_NAMES:
        imgs = DATASETS[ds_name]["images"]
        if kind == "openclip":
            Z = extract_openclip(model, preprocess, imgs, desc=f"{mname[:12]}/{ds_name}")
        else:
            Z = extract_torchvision(model, imgs, desc=f"{mname[:12]}/{ds_name}")
        print(f"  [{ds_name}] Embedding shape: {Z.shape}")
        EMBEDDINGS_BY_DS[ds_name][mname] = Z
        fname = f"{mname.lower().replace(' ','_').replace('/','')}_{ds_name.lower()}.npz"
        np.savez_compressed(OUTDIR/"embeddings"/fname, Z=Z)

    MODEL_NAMES.append(mname)
    del model; torch.cuda.empty_cache(); gc.collect()

if len(MODEL_NAMES) < 3:
    print("\nWARNING: fewer than 3 models loaded successfully -- "
          "cross-model analyses (H2, H3b) will be underpowered.")

# Pixel-PCA baseline (not a foundation model -- a sanity-check floor), per dataset
Z_PIXEL_BY_DS = {}
for ds_name in DATASET_NAMES:
    print(f"\n{'='*55}\n  Pixel-PCA baseline [{ds_name}]\n{'='*55}")
    Z_PIXEL_BY_DS[ds_name] = extract_pixel_pca_features(DATASETS[ds_name]["images"], n_components=64)
    print(f"  Pixel-PCA shape: {Z_PIXEL_BY_DS[ds_name].shape}")

# Backward-compatible flat aliases for the PRIMARY dataset (NIH) -- existing
# H1/H2/H3 statistical cells and Figures 1-6 use these unchanged.
EMBEDDINGS = EMBEDDINGS_BY_DS["NIH"]
Z_pixel    = Z_PIXEL_BY_DS["NIH"]

print(f"\nModels successfully embedded: {MODEL_NAMES}")
print(f"Datasets: {DATASET_NAMES}  (N per dataset: "
      f"{ {ds: DATASETS[ds]['n_total'] for ds in DATASET_NAMES} })")


In [ ]:
# =============================================================================
# CELL 8 — H1 & H2: Demographic Leakage AUROC, Significance, and Cross-Model Diffs
#   Refactored as a reusable function so the identical analysis can run on both
#   NIH and CheXpert without duplicating logic. FDR correction applied to: H1
#   (all model x target tests) and H2's DEMOGRAPHIC comparisons specifically
#   (sex+age, not disease -- disease is a model-quality question, not leakage).
# =============================================================================
import time

TARGET_NAMES = ["sex", "age", "disease"]

def run_h1h2_analysis(ds_name, y_sex, y_age, y_dis, embeddings, z_pixel, model_names):
    targets = {"sex": y_sex, "age": y_age, "disease": y_dis}
    oof_preds, leakage_res = {m: {} for m in model_names}, {m: {} for m in model_names}

    print(f"\n[{ds_name}] H1: Demographic Leakage (per model, per target)\n" + "="*62)
    h1_keys, h1_pvals = [], []
    for mname in model_names:
        Z = embeddings[mname]
        for tname, y in targets.items():
            oof   = probe_cv_oof(Z, y)
            auroc = roc_auc_score(y, oof)
            ci_lo, ci_hi, std = bootstrap_ci_auroc(oof, y)
            p_val = permutation_test_auroc(Z, y, auroc, n_perm=N_PERM)
            oof_preds[mname][tname]   = oof
            leakage_res[mname][tname] = dict(auroc=auroc, ci_lo=ci_lo, ci_hi=ci_hi, std=std, p=p_val)
            h1_keys.append((mname, tname)); h1_pvals.append(p_val)

    h1_fdr_sig = benjamini_hochberg(h1_pvals)
    for (mname, tname), p_val, fdr_ok in zip(h1_keys, h1_pvals, h1_fdr_sig):
        r = leakage_res[mname][tname]
        r["fdr_sig"] = bool(fdr_ok); r["sig"] = "FDR*" if fdr_ok else "ns"
        print(f"  {mname:<16} {tname:<8} AUROC={r['auroc']:.3f} "
              f"[{r['ci_lo']:.3f},{r['ci_hi']:.3f}]  p={p_val:.4f}  {r['sig']}")
    print(f"  -> {h1_fdr_sig.sum()}/{len(h1_fdr_sig)} tests FDR-significant at q={FDR_Q}")

    print(f"\n[{ds_name}] Pixel-PCA baseline\n" + "="*62)
    pixel_res = {}
    for tname, y in targets.items():
        oof   = probe_cv_oof(z_pixel, y)
        auroc = roc_auc_score(y, oof)
        ci_lo, ci_hi, _ = bootstrap_ci_auroc(oof, y)
        pixel_res[tname] = dict(auroc=auroc, ci_lo=ci_lo, ci_hi=ci_hi, oof=oof)
        print(f"  pixel-PCA  {tname:<8} AUROC={auroc:.3f}  [{ci_lo:.3f},{ci_hi:.3f}]")

    h2_results = {}
    if "BioMedCLIP" in model_names:
        print(f"\n[{ds_name}] H2: BioMedCLIP vs. Natural-Pretrained Models\n" + "="*62)
        h2_demo_keys, h2_demo_pvals = [], []
        for tname in targets:
            oof_bio = oof_preds["BioMedCLIP"][tname]
            y = targets[tname]
            h2_results[tname] = {}
            for mname in model_names:
                if mname == "BioMedCLIP": continue
                diff = paired_bootstrap_diff(oof_bio, oof_preds[mname][tname], y)
                h2_results[tname][mname] = diff
                if tname in ("sex", "age"):
                    h2_demo_keys.append((tname, mname)); h2_demo_pvals.append(diff["p"])

        h2_fdr_sig = benjamini_hochberg(h2_demo_pvals) if h2_demo_pvals else np.array([])
        fdr_lookup = {k: s for k, s in zip(h2_demo_keys, h2_fdr_sig)}

        print(f"  -- Demographic comparisons (FDR-corrected, n={len(h2_demo_keys)}) --")
        for tname in ("sex", "age"):
            for mname, diff in h2_results[tname].items():
                fdr_ok = fdr_lookup.get((tname, mname), False)
                diff["fdr_sig"] = bool(fdr_ok)
                direction = "BioMedCLIP > "+mname if diff["mean"]>0 else mname+" > BioMedCLIP"
                tag = "FDR-SIG" if fdr_ok else "ns (post-FDR)"
                print(f"  [{tname:<8}] BioMedCLIP - {mname:<14} = {diff['mean']:+.3f} "
                      f"[{diff['ci_lo']:+.3f},{diff['ci_hi']:+.3f}]  p={diff['p']:.4f}  "
                      f"{tag}  ({direction})")
        n_sig = int(h2_fdr_sig.sum()) if len(h2_fdr_sig) else 0
        print(f"  -> {n_sig}/{len(h2_demo_keys)} demographic comparisons FDR-significant")

        print(f"  -- Disease-detection comparisons (context only, no FDR) --")
        for mname, diff in h2_results["disease"].items():
            direction = "BioMedCLIP > "+mname if diff["mean"]>0 else mname+" > BioMedCLIP"
            print(f"  [disease ] BioMedCLIP - {mname:<14} = {diff['mean']:+.3f} "
                  f"p={diff['p']:.4f}  ({direction})")
    else:
        print(f"\n[{ds_name}] H2 SKIPPED: BioMedCLIP did not load successfully.")

    return leakage_res, pixel_res, h2_results, oof_preds

t0 = time.time()
LEAKAGE_RES, PIXEL_RES, H2_RESULTS, OOF_PREDS = run_h1h2_analysis(
    "NIH", Y_SEX, Y_AGE, Y_DIS, EMBEDDINGS, Z_pixel, MODEL_NAMES)

LEAKAGE_RES_BY_DS = {"NIH": LEAKAGE_RES}
PIXEL_RES_BY_DS   = {"NIH": PIXEL_RES}
H2_RESULTS_BY_DS  = {"NIH": H2_RESULTS}
OOF_PREDS_BY_DS   = {"NIH": OOF_PREDS}

if CHEXPERT_AVAILABLE:
    cx = DATASETS["CheXpert"]
    LEAKAGE_RES_CX, PIXEL_RES_CX, H2_RESULTS_CX, OOF_PREDS_CX = run_h1h2_analysis(
        "CheXpert", cx["sex_bin"], cx["age_bin"], cx["dis_bin"],
        EMBEDDINGS_BY_DS["CheXpert"], Z_PIXEL_BY_DS["CheXpert"], MODEL_NAMES)
    LEAKAGE_RES_BY_DS["CheXpert"] = LEAKAGE_RES_CX
    PIXEL_RES_BY_DS["CheXpert"]   = PIXEL_RES_CX
    H2_RESULTS_BY_DS["CheXpert"]  = H2_RESULTS_CX
    OOF_PREDS_BY_DS["CheXpert"]   = OOF_PREDS_CX

elapsed = (time.time() - t0) / 60
print(f"\nH1/H2 analysis complete for all datasets in {elapsed:.1f} min.")


In [ ]:
# =============================================================================
# CELL 9 — H3a: Calibration Fairness Gap, Raw vs. Prevalence-Matched
#   FDR-corrected across ALL (model, axis) tests in this family, per dataset.
# =============================================================================
def run_h3a_analysis(ds_name, y_dis, y_sex, y_age, oof_preds, model_names):
    calib_res = {m: {} for m in model_names}
    h3a_keys, h3a_pvals = [], []

    print(f"\n[{ds_name}] H3a: Cross-Demographic Calibration Gap\n" + "="*62)
    for mname in model_names:
        oof_d = oof_preds[mname]["disease"]
        for axis_name, group in [("sex", y_sex), ("age", y_age)]:
            res = prevalence_matched_ece_gap(oof_d, y_dis, group)
            idx_A, idx_B = res.pop("idx_A_matched"), res.pop("idx_B_matched")
            p_val = permutation_test_ece_gap(oof_d, y_dis, group, idx_A, idx_B,
                                             res["matched_gap"], n_perm=N_PERM)
            res["p"] = p_val
            res["idx_A_matched"], res["idx_B_matched"] = idx_A, idx_B   # kept for multi-seed cell
            calib_res[mname][axis_name] = res
            h3a_keys.append((mname, axis_name)); h3a_pvals.append(p_val)

    h3a_fdr_sig = benjamini_hochberg(h3a_pvals)
    for (mname, axis_name), p_val, fdr_ok in zip(h3a_keys, h3a_pvals, h3a_fdr_sig):
        r = calib_res[mname][axis_name]
        r["fdr_sig"] = bool(fdr_ok); r["sig"] = "FDR-SIG" if fdr_ok else "ns (post-FDR)"

    last_model = None
    for mname, axis_name in h3a_keys:
        if mname != last_model:
            print(f"\n  {mname}"); last_model = mname
        r = calib_res[mname][axis_name]
        print(f"    [{axis_name}] prevalence A/B={r['prev_A']:.3f}/{r['prev_B']:.3f}  "
              f"raw_gap={r['raw_gap']:.4f}  matched_gap={r['matched_gap']:.4f}  "
              f"(n={r['n_A_matched']}+{r['n_B_matched']})  p={r['p']:.4f}  {r['sig']}")

    n_sig = int(h3a_fdr_sig.sum())
    print(f"\n  -> {n_sig}/{len(h3a_fdr_sig)} calibration tests FDR-significant at q={FDR_Q}")
    return calib_res, n_sig

CALIB_RES, n_h3a_sig_nih = run_h3a_analysis("NIH", Y_DIS, Y_SEX, Y_AGE, OOF_PREDS, MODEL_NAMES)
CALIB_RES_BY_DS = {"NIH": CALIB_RES}

if CHEXPERT_AVAILABLE:
    cx = DATASETS["CheXpert"]
    CALIB_RES_CX, n_h3a_sig_cx = run_h3a_analysis(
        "CheXpert", cx["dis_bin"], cx["sex_bin"], cx["age_bin"], OOF_PREDS_CX, MODEL_NAMES)
    CALIB_RES_BY_DS["CheXpert"] = CALIB_RES_CX

print("\nH3a complete for all datasets.")


In [ ]:
# =============================================================================
# CELL 10 — H3b (EXPLORATORY): Cross-Model Leakage vs. Calibration-Gap
# Age axis (the axis with real H3a signal), per dataset.
# =============================================================================
def run_h3b_analysis(ds_name, leakage_res, calib_res, model_names, axis="age"):
    points = []
    for mname in model_names:
        auroc   = leakage_res[mname][axis]["auroc"]
        gap     = calib_res[mname][axis]["matched_gap"]
        gap_sig = calib_res[mname][axis]["fdr_sig"]
        points.append(dict(model=mname, leakage_auroc=auroc, calib_gap=gap, gap_fdr_sig=gap_sig))
    df_pts = pd.DataFrame(points)

    print(f"\n[{ds_name}] H3b (EXPLORATORY, {axis} axis, n={len(model_names)} models)\n" + "="*62)
    print(df_pts.to_string(index=False))
    n_sig_inputs = int(df_pts["gap_fdr_sig"].sum())
    print(f"  {n_sig_inputs}/{len(df_pts)} underlying gaps individually FDR-significant")

    if len(df_pts) >= 4:
        rho, p = spearmanr(df_pts["leakage_auroc"], df_pts["calib_gap"])
        print(f"  Spearman rho={rho:.3f}  p={p:.3f}")
        if n_sig_inputs == 0:
            print("  WARNING: all underlying gaps non-significant -- this correlation is "
                  "computed across noise and is NOT evidence of a real link.")
        elif n_sig_inputs < len(df_pts):
            print(f"  CAUTION: only {n_sig_inputs}/{len(df_pts)} inputs significant.")
    else:
        rho, p = float("nan"), float("nan")
    return df_pts, rho, p, n_sig_inputs

H3B_AXIS = "age"
df_h3b, H3B_RHO, H3B_P, H3B_N_SIG_INPUTS = run_h3b_analysis(
    "NIH", LEAKAGE_RES, CALIB_RES, MODEL_NAMES, H3B_AXIS)

H3B_BY_DS = {"NIH": dict(df=df_h3b, rho=H3B_RHO, p=H3B_P, n_sig=H3B_N_SIG_INPUTS)}

if CHEXPERT_AVAILABLE:
    df_h3b_cx, rho_cx, p_cx, n_sig_cx = run_h3b_analysis(
        "CheXpert", LEAKAGE_RES_CX, CALIB_RES_CX, MODEL_NAMES, H3B_AXIS)
    H3B_BY_DS["CheXpert"] = dict(df=df_h3b_cx, rho=rho_cx, p=p_cx, n_sig=n_sig_cx)

print("\nH3b complete for all datasets.")


In [ ]:
# =============================================================================
# CELL 10b — H4: Leakage-Direction Ablation (Iterative Nullspace Projection)
#   Converts H3b's correlational evidence into an interventional test: remove
#   the age-decoding direction from embeddings, then ask whether the downstream
#   calibration gap actually shrinks. Run on NIH (the primary, larger dataset).
# =============================================================================
print("H4: Does removing the age-leakage direction reduce the calibration gap?\n" + "="*62)

H4_RESULTS = {}
for mname in MODEL_NAMES:
    Z = EMBEDDINGS[mname]
    ablation  = iterative_nullspace_projection(Z, Y_AGE, seed=SEED)
    Z_ablated = ablation["Z_ablated"]

    auroc_before = LEAKAGE_RES[mname]["age"]["auroc"]
    auroc_after  = ablation["auroc_trace"][-1]
    sanity_ok    = auroc_after <= INLP_AUROC_FLOOR + 0.05

    oof_dis_ablated = probe_cv_oof(Z_ablated, Y_DIS, seed=SEED)
    res_ablated = prevalence_matched_ece_gap(oof_dis_ablated, Y_DIS, Y_AGE, seed=SEED)
    gap_before = CALIB_RES[mname]["age"]["matched_gap"]
    gap_after  = res_ablated["matched_gap"]
    gap_change = gap_after - gap_before
    gap_pct    = (gap_change / gap_before * 100) if gap_before > 1e-9 else float("nan")

    H4_RESULTS[mname] = dict(
        auroc_before=auroc_before, auroc_after=auroc_after,
        auroc_trace=ablation["auroc_trace"], n_directions=ablation["n_directions_removed"],
        sanity_ok=sanity_ok, gap_before=gap_before, gap_after=gap_after,
        gap_change=gap_change, gap_pct_change=gap_pct)

    sanity_tag = "OK" if sanity_ok else "residual leakage still elevated"
    print(f"\n  {mname}")
    print(f"    Directions removed: {ablation['n_directions_removed']}  "
          f"AUROC trace: {[f'{a:.3f}' for a in ablation['auroc_trace']]}")
    print(f"    Leakage AUROC: {auroc_before:.3f} -> {auroc_after:.3f}  [{sanity_tag}]")
    print(f"    Age calib. gap: {gap_before:.4f} -> {gap_after:.4f}  "
          f"({gap_change:+.4f}, {gap_pct:+.1f}%)")

n_shrunk = sum(1 for r in H4_RESULTS.values() if r["gap_change"] < 0)
n_sane   = sum(1 for r in H4_RESULTS.values() if r["sanity_ok"])
print(f"\n  -> Calibration gap shrank after ablation in {n_shrunk}/{len(H4_RESULTS)} models")
print(f"  -> Ablation sanity check (residual AUROC near chance) passed in "
      f"{n_sane}/{len(H4_RESULTS)} models")
if n_sane < len(H4_RESULTS):
    print("  NOTE: where the sanity check failed, age information likely lives in a "
          "higher-dimensional subspace than a single removable direction captures -- "
          "treat that model's gap-change as a lower bound, not a clean causal test.")
print("\nH4 complete.")


In [ ]:
# =============================================================================
# CELL 10c — Jackknife Robustness Check on H3b
#   Tests whether the cross-model correlation is fragile to which specific
#   models were included, by recomputing rho with each model excluded in turn.
# =============================================================================
print("Leave-one-model-out jackknife on H3b's cross-model correlation "
      f"(NIH, {H3B_AXIS} axis)\n" + "="*62)

JACKKNIFE_RESULTS = jackknife_correlation(
    df_h3b["leakage_auroc"].tolist(), df_h3b["calib_gap"].tolist(), df_h3b["model"].tolist())

print(f"  Full-sample rho={H3B_RHO:.3f}  (n={len(MODEL_NAMES)})\n")
for r in JACKKNIFE_RESULTS:
    print(f"  Excluding {r['excluded_model']:<16} -> rho={r['rho']:.3f}  p={r['p']:.3f}  "
          f"(n={r['n_remaining']})")

rhos = [r["rho"] for r in JACKKNIFE_RESULTS if not np.isnan(r["rho"])]
if rhos:
    print(f"\n  Jackknife rho range: [{min(rhos):.3f}, {max(rhos):.3f}]  "
          f"(full-sample rho={H3B_RHO:.3f})")
    if min(rhos) > 0.5:
        print("  All leave-one-out correlations remain strongly positive -- the H3b "
              "finding is not driven by any single model.")
    else:
        print("  CAUTION: at least one leave-one-out correlation drops substantially -- "
              "the H3b finding may be sensitive to a specific model's inclusion.")
print("\nJackknife complete.")


In [ ]:
# =============================================================================
# CELL 10d — Continuous Age Dose-Response (replaces the binary median split)
#   Tests whether the calibration gap scales smoothly with age, addressing the
#   "arbitrary binary threshold" objection to H3a's age-axis finding.
# =============================================================================
print(f"Age dose-response: calibration gap across {N_AGE_QUANTILES} age quantiles (NIH)\n" + "="*62)

DOSE_RESPONSE_RESULTS = {}
for mname in MODEL_NAMES:
    oof_dis = OOF_PREDS[mname]["disease"]
    bins = quartile_age_dose_response(oof_dis, Y_DIS, Y_AGE_CONT, n_quantiles=N_AGE_QUANTILES)
    DOSE_RESPONSE_RESULTS[mname] = bins
    print(f"\n  {mname}")
    for b in bins:
        print(f"    Q{b['quantile']} (age {b['age_lo']:.0f}-{b['age_hi']:.0f}, n={b['n']}): "
              f"ECE={b['ece']:.4f}  gap_vs_lowest={b['gap_vs_lowest_quantile']:+.4f}")

print("\n  Monotonicity check (does ECE increase with age, not just jump at one threshold?):")
for mname, bins in DOSE_RESPONSE_RESULTS.items():
    eces = [b["ece"] for b in bins]
    is_monotonic = all(eces[i] <= eces[i+1] + 1e-9 for i in range(len(eces)-1))
    print(f"    {mname:<16} ECE sequence: {[f'{e:.3f}' for e in eces]}  "
          f"monotonic increasing: {is_monotonic}")
print("\nDose-response analysis complete.")


In [ ]:
# =============================================================================
# CELL 10e — Multi-Seed Stability Check (the ResNet-50 raw-to-matched swing)
#   Re-runs prevalence-matching with multiple seeds to test whether the matched
#   gap estimate is stable, or sensitive to exactly which patients are dropped.
# =============================================================================
print(f"Multi-seed stability check ({N_MATCH_SEEDS} seeds) on prevalence-matched gaps (NIH)\n" + "="*62)

MULTISEED_RESULTS = {m: {} for m in MODEL_NAMES}
for mname in MODEL_NAMES:
    oof_dis = OOF_PREDS[mname]["disease"]
    for axis_name, group in [("sex", Y_SEX), ("age", Y_AGE)]:
        stab = multiseed_matched_gap(oof_dis, Y_DIS, group, n_seeds=N_MATCH_SEEDS)
        MULTISEED_RESULTS[mname][axis_name] = stab
        orig_gap = CALIB_RES[mname][axis_name]["matched_gap"]
        cv = stab["std"] / stab["mean"] if stab["mean"] > 1e-9 else float("nan")
        print(f"  {mname:<16} [{axis_name}] single-seed={orig_gap:.4f}  "
              f"{N_MATCH_SEEDS}-seed mean={stab['mean']:.4f}  std={stab['std']:.4f}  "
              f"range=[{stab['min']:.4f},{stab['max']:.4f}]  CV={cv:.2f}")

print("\n  Focused check: ResNet-50's age-axis gap (the largest raw-to-matched swing observed)")
if "ResNet-50" in MULTISEED_RESULTS:
    r = MULTISEED_RESULTS["ResNet-50"]["age"]
    print(f"    Mean={r['mean']:.4f}  std={r['std']:.4f}")
    print(f"    All seeds: {[f'{g:.4f}' for g in r['all_gaps']]}")
    cv50 = r["std"] / max(r["mean"], 1e-9)
    if cv50 < 0.25:
        print(f"    Stable across resampling seeds (CV={cv50:.2f} < 0.25) -- the large "
              "raw-to-matched swing is a genuine property of this model, not a matching artifact.")
    else:
        print(f"    CAUTION: high variance across seeds (CV={cv50:.2f}) -- report the seed "
              "range rather than a single point estimate for this pair.")
print("\nMulti-seed stability check complete.")


In [ ]:
# =============================================================================
# CELL 11 — Figure 1: Demographic Leakage AUROC (per model) with CI + Significance
# =============================================================================
COLORS  = PALETTE[:len(MODEL_NAMES)]
TARGET_NAMES = ["sex", "age", "disease"]

n_m = len(MODEL_NAMES)
fig, axes = plt.subplots(1, 3, figsize=(FIG_W_DOUBLE, 3.0), sharey=True,
                         gridspec_kw=dict(wspace=0.12))

chance_line, pixel_line = None, None
for ai, tname in enumerate(TARGET_NAMES):
    ax = axes[ai]
    aurocs = [LEAKAGE_RES[m][tname]["auroc"] for m in MODEL_NAMES]
    los    = [LEAKAGE_RES[m][tname]["ci_lo"] for m in MODEL_NAMES]
    his    = [LEAKAGE_RES[m][tname]["ci_hi"] for m in MODEL_NAMES]
    errs   = [[a-lo for a,lo in zip(aurocs,los)], [hi-a for a,hi in zip(aurocs,his)]]
    sigs   = [LEAKAGE_RES[m][tname]["sig"] for m in MODEL_NAMES]

    bars = ax.bar(range(n_m), aurocs, yerr=errs, color=COLORS, capsize=3,
                  zorder=3, edgecolor="white", linewidth=0.6,
                  error_kw=dict(lw=0.8))
    chance_line = ax.axhline(0.5, color="gray", lw=0.9, ls="--", zorder=2)
    pixel_line  = ax.axhline(PIXEL_RES[tname]["auroc"], color="black", lw=0.9, ls=":", zorder=2)

    for b, s in zip(bars, sigs):
        if s == "FDR*":
            ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.022, "*",
                   ha="center", fontsize=11, fontweight="bold", color="darkgreen", zorder=4)

    # Panel tag in the corner -- not a title, just disambiguates the 3 panels
    ax.text(0.04, 0.96, tname, transform=ax.transAxes, fontsize=8, fontweight="bold",
            ha="left", va="top",
            bbox=dict(boxstyle="round,pad=0.2", facecolor="white", edgecolor="none", alpha=0.75))

    ax.set_xticks(range(n_m))
    ax.set_xticklabels(MODEL_NAMES, fontsize=6, rotation=35, ha="right", rotation_mode="anchor")
    if ai == 0: ax.set_ylabel("Probe AUROC (5-fold CV, OOF)")
    ax.set_ylim(0.35, 1.08)
    ax.grid(axis="y", lw=0.3, alpha=0.4, zorder=0)
    _despine(ax)

fig.legend([chance_line, pixel_line], ["Chance", "Pixel-PCA baseline"],
          frameon=False, fontsize=6.5, ncol=2, loc="upper center",
          bbox_to_anchor=(0.5, 1.0))
fig.tight_layout(rect=[0, 0, 1, 0.93])
_savefig(fig, "fig1_leakage_auroc", double=True)
plt.show()
plt.close(fig)
print("Figure 1 saved.")
print(f"Caption (for paper): Demographic leakage AUROC per model and target. "
      f"* = FDR-significant at q={FDR_Q} (Benjamini-Hochberg across all 15 tests).")


In [ ]:
# =============================================================================
# CELL 12 — Figure 2 (H2 KEY): Forest Plot — BioMedCLIP minus Baselines
# =============================================================================
if "BioMedCLIP" in MODEL_NAMES and len(H2_RESULTS) > 0:
    other_models = [m for m in MODEL_NAMES if m != "BioMedCLIP"]
    fig, axes = plt.subplots(1, len(TARGET_NAMES), figsize=(FIG_W_DOUBLE, 0.55*len(other_models)+1.3),
                             sharex=True, sharey=True, gridspec_kw=dict(wspace=0.08))

    panel_labels = {"sex": "sex (FDR)", "age": "age (FDR)", "disease": "disease (context)"}
    for ai, tname in enumerate(TARGET_NAMES):
        ax = axes[ai]
        means = [H2_RESULTS[tname][m]["mean"]  for m in other_models]
        los   = [H2_RESULTS[tname][m]["ci_lo"] for m in other_models]
        his   = [H2_RESULTS[tname][m]["ci_hi"] for m in other_models]
        fdr_flags = [H2_RESULTS[tname][m].get("fdr_sig", H2_RESULTS[tname][m]["p"] < 0.05)
                    for m in other_models]

        ys = np.arange(len(other_models))
        for y, mean, lo, hi, sig, c in zip(ys, means, los, his, fdr_flags, COLORS[1:]):
            ax.plot([lo, hi], [y, y], color=c, lw=1.6, zorder=2)
            marker = "o" if sig else "s"
            ax.scatter([mean], [y], color=c, s=30, zorder=3, marker=marker,
                      edgecolor="black", linewidth=0.4)
        ax.axvline(0, color="black", lw=0.8, ls="-", zorder=1)
        ax.set_yticks(ys)
        if ai == 0:
            ax.set_yticklabels(other_models, fontsize=6.5)
        ax.margins(y=0.25)
        ax.set_xlabel(r"$\Delta$AUROC vs. BioMedCLIP", fontsize=6)
        ax.text(0.06, 0.04, panel_labels[tname], transform=ax.transAxes, fontsize=7,
                fontweight="bold", ha="left", va="bottom",
                bbox=dict(boxstyle="round,pad=0.2", facecolor="white", edgecolor="none", alpha=0.85))
        ax.grid(axis="x", lw=0.3, alpha=0.4)
        _despine(ax)

    leg_handles = [
        plt.Line2D([0],[0], marker="o", color="w", markerfacecolor="gray",
                   markeredgecolor="black", markersize=6, label="FDR-significant"),
        plt.Line2D([0],[0], marker="s", color="w", markerfacecolor="gray",
                   markeredgecolor="black", markersize=6, label="Not significant"),
    ]
    fig.legend(handles=leg_handles, frameon=False, fontsize=6.5, ncol=2,
              loc="upper center", bbox_to_anchor=(0.5, 1.02))
    fig.tight_layout(rect=[0, 0, 1, 0.90])
    _savefig(fig, "fig2_h2_forest_plot", double=True)
    plt.show()
    plt.close(fig)
    print("Figure 2 saved.")
    print("Caption (for paper): H2 -- paired-bootstrap AUROC difference, BioMedCLIP vs. "
          "natural-pretrained baselines. Circle = FDR-significant on sex/age (q={:.2f}); "
          "disease panel is model-quality context, not part of the leakage claim.".format(FDR_Q))
else:
    print("Figure 2 SKIPPED: BioMedCLIP unavailable.")


In [ ]:
# =============================================================================
# CELL 13 — Figure 3: Foundation-Model Embeddings vs. Raw-Pixel-PCA Baseline
# Shows whether leakage is amplified beyond what's trivially visible in pixels
# =============================================================================
fig, axes = plt.subplots(1, 3, figsize=(FIG_W_DOUBLE, 3.0), sharey=True,
                         gridspec_kw=dict(wspace=0.12))

chance_line, pixel_line = None, None
for ai, tname in enumerate(TARGET_NAMES):
    ax = axes[ai]
    fm_aurocs = [LEAKAGE_RES[m][tname]["auroc"] for m in MODEL_NAMES]
    px_auroc  = PIXEL_RES[tname]["auroc"]

    xs = np.arange(n_m)
    ax.bar(xs, fm_aurocs, color=COLORS, width=0.6, zorder=3,
          edgecolor="white", linewidth=0.6)
    pixel_line  = ax.axhline(px_auroc, color="black", lw=1.3, ls=":", zorder=4)
    chance_line = ax.axhline(0.5, color="gray", lw=0.8, ls="--", zorder=2)

    for x, v in zip(xs, fm_aurocs):
        amp = v - px_auroc
        ax.text(x, v+0.025, f"{amp:+.2f}", ha="center", fontsize=6,
                color="darkgreen" if amp > 0 else "darkred", zorder=5)

    ax.text(0.04, 0.96, tname, transform=ax.transAxes, fontsize=8, fontweight="bold",
            ha="left", va="top",
            bbox=dict(boxstyle="round,pad=0.2", facecolor="white", edgecolor="none", alpha=0.75))

    ax.set_xticks(xs); ax.set_xticklabels(MODEL_NAMES, fontsize=6, rotation=35, ha="right", rotation_mode="anchor")
    if ai == 0: ax.set_ylabel("AUROC")
    ax.set_ylim(0.35, 1.12)
    _despine(ax)

fig.legend([chance_line, pixel_line], ["Chance", "Pixel-PCA baseline"],
          frameon=False, fontsize=6.5, ncol=2, loc="upper center",
          bbox_to_anchor=(0.5, 1.0))
fig.tight_layout(rect=[0, 0, 1, 0.92])
_savefig(fig, "fig3_amplification_vs_pixels", double=True)
plt.show()
plt.close(fig)
print("Figure 3 saved.")
print("Caption (for paper): Amplification beyond raw pixel content. Numbers above bars "
      "= AUROC delta vs. pixel-PCA baseline; positive = model encodes more than raw "
      "pixels alone reveal.")


In [ ]:
# =============================================================================
# CELL 14 — Figure 4: Reliability Diagrams by Demographic Subgroup
# =============================================================================
def _reliability_curve(probs, labels, n_bins=10):
    bins  = np.linspace(0.0, 1.0, n_bins + 1)
    centers, accs, confs, counts = [], [], [], []
    for lo, hi in zip(bins[:-1], bins[1:]):
        mask = (probs >= lo) & (probs < hi) if hi < 1.0 else (probs >= lo) & (probs <= hi)
        if mask.sum() == 0:
            continue
        centers.append((lo+hi)/2); accs.append(labels[mask].mean())
        confs.append(probs[mask].mean()); counts.append(mask.sum())
    return np.array(centers), np.array(accs), np.array(confs), np.array(counts)

# Prefer a model with an FDR-significant age-axis calibration gap (illustrates
# the actual finding) over an arbitrary "best AUROC" choice.
sig_age_models = [m for m in MODEL_NAMES if CALIB_RES[m]["age"]["fdr_sig"]]
if sig_age_models:
    best_model = max(sig_age_models, key=lambda m: CALIB_RES[m]["age"]["matched_gap"])
else:
    best_model = max(MODEL_NAMES, key=lambda m: LEAKAGE_RES[m]["disease"]["auroc"])
oof_best = OOF_PREDS[best_model]["disease"]

fig, axes = plt.subplots(1, 2, figsize=(FIG_W_DOUBLE, 3.0))

for ai, (axis_name, group) in enumerate([("sex", Y_SEX), ("age", Y_AGE)]):
    ax = axes[ai]
    ax.plot([0,1], [0,1], color="gray", lw=0.8, ls="--", label="Perfect calibration")

    for gv, lbl, color in [(0, f"{axis_name}=0", PALETTE[0]), (1, f"{axis_name}=1", PALETTE[3])]:
        mask = group == gv
        c, a, f_, n = _reliability_curve(oof_best[mask], Y_DIS[mask])
        ece_g = compute_ece(oof_best[mask], Y_DIS[mask])
        ax.plot(f_, a, marker="o", ms=3.5, color=color, lw=1.2,
               label=f"{lbl}  (ECE={ece_g:.3f}, n={mask.sum()})")

    ax.text(0.96, 0.04, axis_name, transform=ax.transAxes, fontsize=8, fontweight="bold",
            ha="right", va="bottom",
            bbox=dict(boxstyle="round,pad=0.2", facecolor="white", edgecolor="none", alpha=0.75))
    ax.set_xlabel("Mean predicted probability")
    ax.set_ylabel("Empirical accuracy" if ai == 0 else "")
    ax.set_xlim(0,1); ax.set_ylim(0,1)
    ax.legend(frameon=False, fontsize=6, loc="upper left")
    _despine(ax)

fig.tight_layout()
_savefig(fig, "fig4_reliability_diagrams", double=True)
plt.show()
plt.close(fig)
print("Figure 4 saved.")
print(f"Caption (for paper): Reliability diagrams for {best_model}'s disease-detection "
      f"probe, by sex and by age subgroup.")


In [ ]:
# =============================================================================
# CELL 15 — Figure 5: ECE Gap — Raw vs. Prevalence-Matched (H3a key figure)
# =============================================================================
fig, axes = plt.subplots(1, 2, figsize=(FIG_W_DOUBLE, 3.0), gridspec_kw=dict(wspace=0.15))

for ai, axis_name in enumerate(["sex", "age"]):
    ax = axes[ai]
    raw_gaps     = [CALIB_RES[m][axis_name]["raw_gap"]     for m in MODEL_NAMES]
    matched_gaps = [CALIB_RES[m][axis_name]["matched_gap"] for m in MODEL_NAMES]
    fdr_flags    = [CALIB_RES[m][axis_name]["fdr_sig"]     for m in MODEL_NAMES]

    xs = np.arange(n_m)
    w  = 0.35
    ax.bar(xs - w/2, raw_gaps,     width=w, color=PALETTE[0], label="Raw gap",
          zorder=3, edgecolor="white", linewidth=0.5)
    ax.bar(xs + w/2, matched_gaps, width=w, color=PALETTE[3], label="Prevalence-matched",
          zorder=3, edgecolor="white", linewidth=0.5)

    max_h = max(raw_gaps + matched_gaps) if (raw_gaps + matched_gaps) else 0.01
    for x, v, sig in zip(xs, matched_gaps, fdr_flags):
        if sig:
            ax.text(x + w/2, v + max_h*0.05, "*", ha="center", fontsize=11,
                   fontweight="bold", color="darkgreen", zorder=4)

    ax.text(0.04, 0.96, axis_name, transform=ax.transAxes, fontsize=8, fontweight="bold",
            ha="left", va="top",
            bbox=dict(boxstyle="round,pad=0.2", facecolor="white", edgecolor="none", alpha=0.75))
    ax.set_xticks(xs); ax.set_xticklabels(MODEL_NAMES, fontsize=6, rotation=35, ha="right", rotation_mode="anchor")
    if ai == 0: ax.set_ylabel("|ECE(group A) - ECE(group B)|")
    ax.set_ylim(0, max_h * 1.25)
    ax.legend(frameon=False, fontsize=6, loc="upper right")
    ax.grid(axis="y", lw=0.3, alpha=0.4, zorder=0)
    _despine(ax)

fig.tight_layout()
_savefig(fig, "fig5_ece_gap_matched", double=True)
plt.show()
plt.close(fig)

n_sig_total = sum(CALIB_RES[m][a]["fdr_sig"] for m in MODEL_NAMES for a in ("sex","age"))
print("Figure 5 saved.")
print(f"Caption (for paper): Calibration gap after prevalence matching. "
      f"* = FDR-significant at q={FDR_Q} (Benjamini-Hochberg across all {n_m*2} tests); "
      f"{n_sig_total}/{n_m*2} survive correction.")


In [ ]:
# =============================================================================
# CELL 16 — Figure 6 (EXPLORATORY): Cross-Model Leakage vs. Calibration Gap
# Now on the AGE axis (where H3a found real signal), not sex (a clean null).
# =============================================================================
fig, ax = plt.subplots(figsize=(FIG_W_SINGLE, 3.4))

for i, row in df_h3b.iterrows():
    sig = row["gap_fdr_sig"]
    marker = "o" if sig else "x"
    kw = dict(edgecolor="black", linewidth=0.5) if sig else dict(linewidth=1.5)
    ax.scatter(row["leakage_auroc"], row["calib_gap"], s=75, marker=marker,
              color=COLORS[MODEL_NAMES.index(row["model"])], zorder=3, **kw)
    ax.annotate(row["model"], (row["leakage_auroc"], row["calib_gap"]),
               fontsize=6, xytext=(5,5), textcoords="offset points")

if len(df_h3b) >= 2:
    coef = np.polyfit(df_h3b["leakage_auroc"], df_h3b["calib_gap"], 1)
    xs   = np.linspace(df_h3b["leakage_auroc"].min(), df_h3b["leakage_auroc"].max(), 50)
    ax.plot(xs, np.polyval(coef, xs), color="gray", lw=1.0, ls="--", alpha=0.7, zorder=1)

ax.margins(x=0.18, y=0.18)   # headroom so point-label annotations don't clip at the edges
ax.set_xlabel(f"{H3B_AXIS.capitalize()}-leakage AUROC")
ax.set_ylabel(f"Calibration gap (matched, {H3B_AXIS} axis)")
ax.text(0.96, 0.04, "o = FDR-sig gap, x = not", transform=ax.transAxes, fontsize=6,
        ha="right", va="bottom", style="italic", color="dimgray")
_despine(ax)
fig.tight_layout()
_savefig(fig, "fig6_exploratory_crossmodel")
plt.show()
plt.close(fig)

print("Figure 6 saved.")
print(f"Caption (for paper, EXPLORATORY only): Cross-model {H3B_AXIS}-leakage AUROC vs. "
      f"{H3B_AXIS}-axis calibration gap. Spearman rho={H3B_RHO:.2f}, p={H3B_P:.2f} (n={len(df_h3b)}); "
      f"{H3B_N_SIG_INPUTS}/{len(df_h3b)} underlying gaps individually FDR-significant.")
if H3B_N_SIG_INPUTS == 0:
    print("Every point above is an 'x' -- none of the underlying calibration gaps are "
          "individually significant. The dashed trend line should NOT be interpreted "
          "as evidence of a real relationship.")
elif H3B_N_SIG_INPUTS < len(df_h3b):
    print(f"Only {H3B_N_SIG_INPUTS}/{len(df_h3b)} points rest on individually-significant "
          "gaps -- interpret the trend line with real caution.")


In [ ]:
# =============================================================================
# CELL 16b — Figure 7: Cross-Dataset Replication (NIH vs. CheXpert)
# =============================================================================
if CHEXPERT_AVAILABLE:
    other_models = [m for m in MODEL_NAMES if m != "BioMedCLIP"]
    fig, axes = plt.subplots(1, 2, figsize=(FIG_W_DOUBLE, 3.2), gridspec_kw=dict(wspace=0.28))

    # Left: H2 sex+age deltas, NIH vs CheXpert
    ax = axes[0]
    xs = np.arange(len(other_models))
    w  = 0.2
    offsets  = [-1.5*w, -0.5*w, 0.5*w, 1.5*w]
    bar_defs = [("sex","NIH"), ("sex","CheXpert"), ("age","NIH"), ("age","CheXpert")]
    colors4  = [PALETTE[0], PALETTE[1], PALETTE[3], PALETTE[4]]
    for off, (axis_n, ds), c in zip(offsets, bar_defs, colors4):
        vals = [H2_RESULTS_BY_DS[ds][axis_n][m]["mean"] for m in other_models]
        ax.bar(xs+off, vals, width=w, color=c, label=f"{axis_n}/{ds}",
              edgecolor="white", linewidth=0.4, zorder=3)
    ax.axhline(0, color="black", lw=0.8)
    ax.set_xticks(xs); ax.set_xticklabels(other_models, fontsize=6, rotation=35, ha="right",
                                          rotation_mode="anchor")
    ax.set_ylabel(r"$\Delta$AUROC vs. BioMedCLIP")
    ax.text(0.04, 0.96, "H2 replication", transform=ax.transAxes, fontsize=7, fontweight="bold",
            ha="left", va="top", bbox=dict(boxstyle="round,pad=0.2", facecolor="white",
                                           edgecolor="none", alpha=0.8))
    ax.legend(frameon=False, fontsize=5.5, ncol=2, loc="lower right")
    ax.grid(axis="y", lw=0.3, alpha=0.4, zorder=0)
    _despine(ax)

    # Right: H3a age-axis matched gap, NIH vs CheXpert
    ax = axes[1]
    xs2 = np.arange(len(MODEL_NAMES))
    w2  = 0.35
    nih_gaps = [CALIB_RES_BY_DS["NIH"][m]["age"]["matched_gap"] for m in MODEL_NAMES]
    cx_gaps  = [CALIB_RES_BY_DS["CheXpert"][m]["age"]["matched_gap"] for m in MODEL_NAMES]
    ax.bar(xs2-w2/2, nih_gaps, width=w2, color=PALETTE[0], label="NIH",
          edgecolor="white", linewidth=0.4, zorder=3)
    ax.bar(xs2+w2/2, cx_gaps,  width=w2, color=PALETTE[4], label="CheXpert",
          edgecolor="white", linewidth=0.4, zorder=3)
    ax.set_xticks(xs2); ax.set_xticklabels(MODEL_NAMES, fontsize=6, rotation=35, ha="right",
                                           rotation_mode="anchor")
    ax.set_ylabel("Age-axis matched ECE gap")
    ax.text(0.96, 0.96, "H3a replication", transform=ax.transAxes, fontsize=7, fontweight="bold",
            ha="right", va="top", bbox=dict(boxstyle="round,pad=0.2", facecolor="white",
                                           edgecolor="none", alpha=0.8))
    ax.legend(frameon=False, fontsize=6, loc="upper left")
    ax.grid(axis="y", lw=0.3, alpha=0.4, zorder=0)
    _despine(ax)

    fig.tight_layout()
    _savefig(fig, "fig7_crossdataset_replication", double=True)
    plt.show()
    plt.close(fig)
    print("Figure 7 saved.")
    print("Caption (for paper): Cross-dataset replication. Left: H2 demographic-leakage "
          "deltas (BioMedCLIP vs. baselines) on NIH vs. CheXpert. Right: H3a age-axis "
          "matched calibration gap on NIH vs. CheXpert.")
else:
    print("Figure 7 SKIPPED: CheXpert not attached.")


In [ ]:
# =============================================================================
# CELL 16c — Figure 8: Replication-Consistency Scatter
# One point per (model, axis) H2 comparison; x=NIH effect, y=CheXpert effect.
# Points near the diagonal indicate consistent replication.
# =============================================================================
if CHEXPERT_AVAILABLE:
    other_models = [m for m in MODEL_NAMES if m != "BioMedCLIP"]
    fig, ax = plt.subplots(figsize=(FIG_W_SINGLE, 4.0))

    markers = {"sex": "o", "age": "^"}
    pts_x, pts_y = [], []
    for axis_n in ("sex", "age"):
        for mi, mname in enumerate(other_models):
            x = H2_RESULTS_BY_DS["NIH"][axis_n][mname]["mean"]
            y = H2_RESULTS_BY_DS["CheXpert"][axis_n][mname]["mean"]
            pts_x.append(x); pts_y.append(y)
            ax.scatter(x, y, s=60, marker=markers[axis_n], color=COLORS[(mi+1) % len(COLORS)],
                      edgecolor="black", linewidth=0.4, zorder=3)

    lo = min(pts_x + pts_y) - 0.02
    hi = max(pts_x + pts_y) + 0.02
    ax.plot([lo, hi], [lo, hi], color="gray", lw=1.0, ls="--", zorder=1)
    ax.axhline(0, color="black", lw=0.6, zorder=1)
    ax.axvline(0, color="black", lw=0.6, zorder=1)
    ax.set_xlim(lo, hi); ax.set_ylim(lo, hi)

    leg_handles = [
        plt.Line2D([0],[0], marker="o", color="w", markerfacecolor="gray",
                   markeredgecolor="black", markersize=7, label="sex"),
        plt.Line2D([0],[0], marker="^", color="w", markerfacecolor="gray",
                   markeredgecolor="black", markersize=7, label="age"),
    ]
    ax.legend(handles=leg_handles, frameon=False, fontsize=7, loc="upper left")
    ax.set_xlabel(r"$\Delta$AUROC vs. BioMedCLIP (NIH)")
    ax.set_ylabel(r"$\Delta$AUROC vs. BioMedCLIP (CheXpert)")
    ax.text(0.96, 0.04, "dashed line = perfect replication", transform=ax.transAxes,
            fontsize=6, ha="right", va="bottom", style="italic", color="dimgray")
    _despine(ax)
    fig.tight_layout()
    _savefig(fig, "fig8_replication_consistency")
    plt.show()
    plt.close(fig)

    corr_consistency = np.corrcoef(pts_x, pts_y)[0,1] if len(pts_x) >= 2 else float("nan")
    print("Figure 8 saved.")
    print(f"Caption (for paper): Replication consistency of H2 effects across datasets. "
          f"Pearson r between NIH and CheXpert effect sizes = {corr_consistency:.2f} (n={len(pts_x)}).")
else:
    print("Figure 8 SKIPPED: CheXpert not attached.")


In [ ]:
# =============================================================================
# CELL 16d — Figure 9: H4 Ablation — Calibration Gap and Sanity Check
# =============================================================================
fig, axes = plt.subplots(1, 2, figsize=(FIG_W_DOUBLE, 3.2), gridspec_kw=dict(wspace=0.3))

# Left: matched gap before vs after ablation
ax = axes[0]
xs = np.arange(len(MODEL_NAMES))
w  = 0.35
before = [H4_RESULTS[m]["gap_before"] for m in MODEL_NAMES]
after  = [H4_RESULTS[m]["gap_after"]  for m in MODEL_NAMES]
ax.bar(xs-w/2, before, width=w, color=PALETTE[3], label="Before ablation",
      edgecolor="white", linewidth=0.4, zorder=3)
ax.bar(xs+w/2, after,  width=w, color=PALETTE[2], label="After ablation",
      edgecolor="white", linewidth=0.4, zorder=3)
ax.set_xticks(xs); ax.set_xticklabels(MODEL_NAMES, fontsize=6, rotation=35, ha="right",
                                      rotation_mode="anchor")
ax.set_ylabel("Age-axis matched ECE gap")
ax.text(0.04, 0.96, "calibration gap", transform=ax.transAxes, fontsize=7, fontweight="bold",
        ha="left", va="top", bbox=dict(boxstyle="round,pad=0.2", facecolor="white",
                                       edgecolor="none", alpha=0.8))
ax.legend(frameon=False, fontsize=6.5, loc="upper right")
ax.grid(axis="y", lw=0.3, alpha=0.4, zorder=0)
_despine(ax)

# Right: AUROC trace (sanity check) -- does leakage actually drop near chance?
ax = axes[1]
for mi, mname in enumerate(MODEL_NAMES):
    trace = H4_RESULTS[mname]["auroc_trace"]
    ax.plot(range(len(trace)), trace, marker="o", ms=3, lw=1.2,
           color=COLORS[mi % len(COLORS)], label=mname, zorder=3)
ax.axhline(0.5, color="gray", lw=0.8, ls="--", zorder=1)
ax.axhline(INLP_AUROC_FLOOR, color="black", lw=0.8, ls=":", zorder=1)
ax.set_xlabel("Directions removed")
ax.set_ylabel("Residual age-leakage AUROC")
ax.text(0.04, 0.96, "sanity check", transform=ax.transAxes, fontsize=7, fontweight="bold",
        ha="left", va="top", bbox=dict(boxstyle="round,pad=0.2", facecolor="white",
                                       edgecolor="none", alpha=0.8))
ax.legend(frameon=False, fontsize=5.5, loc="upper right")
ax.grid(axis="y", lw=0.3, alpha=0.4, zorder=0)
_despine(ax)

fig.tight_layout()
_savefig(fig, "fig9_ablation_before_after", double=True)
plt.show()
plt.close(fig)
print("Figure 9 saved.")
print("Caption (for paper): H4 ablation. Left: age-axis calibration gap before vs. after "
      "removing the age-decoding direction(s). Right: sanity check showing residual "
      "leakage AUROC converging toward chance (dashed) as directions are removed; "
      "dotted line marks the stopping threshold.")


In [ ]:
# =============================================================================
# CELL 16e — Figure 10: Jackknife Robustness on H3b Correlation
# =============================================================================
fig, ax = plt.subplots(figsize=(FIG_W_SINGLE, 3.2))

ys = np.arange(len(JACKKNIFE_RESULTS))
rhos_jk = [r["rho"] for r in JACKKNIFE_RESULTS]
labels_jk = [f"excl. {r['excluded_model']}" for r in JACKKNIFE_RESULTS]

ax.axvline(H3B_RHO, color="black", lw=1.0, ls="-", zorder=1)
ax.scatter(rhos_jk, ys, s=50, color=PALETTE[4], edgecolor="black", linewidth=0.5, zorder=3)
ax.set_yticks(ys); ax.set_yticklabels(labels_jk, fontsize=6.5)
ax.set_xlim(-1.05, 1.05)
ax.set_xlabel(r"Leave-one-out Spearman $\rho$")
ax.margins(y=0.15)
ax.text(0.97, 0.04, f"solid line = full-sample rho ({H3B_RHO:.2f})", transform=ax.transAxes,
        fontsize=6, ha="right", va="bottom", style="italic", color="dimgray")
_despine(ax)
fig.tight_layout()
_savefig(fig, "fig10_jackknife_robustness")
plt.show()
plt.close(fig)
print("Figure 10 saved.")
print("Caption (for paper): Leave-one-model-out jackknife on the H3b cross-model "
      "correlation. Each point recomputes rho with one model excluded; the vertical "
      "line marks the full-sample rho.")


In [ ]:
# =============================================================================
# CELL 16f — Figure 11: Age Dose-Response (quartile ECE, not just binary split)
# =============================================================================
fig, ax = plt.subplots(figsize=(FIG_W_SINGLE, 3.4))

for mi, mname in enumerate(MODEL_NAMES):
    bins = DOSE_RESPONSE_RESULTS[mname]
    xs   = [b["age_mean"] for b in bins]
    eces = [b["ece"] for b in bins]
    ax.plot(xs, eces, marker="o", ms=4, lw=1.3, color=COLORS[mi % len(COLORS)], label=mname,
           zorder=3)

ax.set_xlabel("Mean age in quantile bin")
ax.set_ylabel("ECE (disease-detection probe)")
ax.text(0.04, 0.96, f"{N_AGE_QUANTILES} age quantiles", transform=ax.transAxes, fontsize=7,
        fontweight="bold", ha="left", va="top",
        bbox=dict(boxstyle="round,pad=0.2", facecolor="white", edgecolor="none", alpha=0.8))
ax.legend(frameon=False, fontsize=6, loc="lower right")
ax.grid(axis="y", lw=0.3, alpha=0.4)
_despine(ax)
fig.tight_layout()
_savefig(fig, "fig11_age_dose_response")
plt.show()
plt.close(fig)
print("Figure 11 saved.")
print("Caption (for paper): Calibration error as a function of age quantile (not a single "
      "binary split). A smoothly increasing curve indicates the age-calibration gap is a "
      "genuine dose-response relationship rather than an artifact of the median-split threshold.")


In [ ]:
# =============================================================================
# CELL 17 — Tables (booktabs LaTeX + CSV)
# =============================================================================
B  = chr(92); LB = chr(123); RB = chr(125); BB = B + B

def cmd(name, arg=None):
    return B + name + (LB + str(arg) + RB if arg is not None else "")

def write_booktabs(fpath, df, caption, label, col_fmt, group_col=None):
    header = " & ".join(cmd("textbf", c) for c in df.columns) + " " + BB
    rows, prev = [], None
    for _, row in df.iterrows():
        cur = row.get(group_col, None) if group_col else None
        if cur is not None and cur != prev and prev is not None:
            rows.append("    " + cmd("addlinespace") + "[2pt]")
        prev = cur
        rows.append("    " + " & ".join(str(v) for v in row) + " " + BB)
    lines = [
        cmd("begin","table") + "[t]", "  " + cmd("centering"), "  " + cmd("small"),
        "  " + B+"caption"+LB+caption+RB, "  " + B+"label"+LB+label+RB,
        "  " + cmd("begin","tabular") + LB+col_fmt+RB,
        "    " + cmd("toprule"), "    " + header, "    " + cmd("midrule"),
        *rows, "    " + cmd("bottomrule"), "  " + cmd("end","tabular"), cmd("end","table"),
    ]
    Path(fpath).write_text("\n".join(lines), encoding="utf-8")

# ── Table 1: Leakage AUROC (H1), all datasets ────────────────────────────────
rows1 = []
for ds_name in DATASET_NAMES:
    leak, pix = LEAKAGE_RES_BY_DS[ds_name], PIXEL_RES_BY_DS[ds_name]
    for tname in TARGET_NAMES:
        rows1.append({"Dataset":ds_name, "Model":"Pixel-PCA (baseline)", "Target":tname,
                      "AUROC":f"{pix[tname]['auroc']:.3f}",
                      "95% CI":f"[{pix[tname]['ci_lo']:.3f},{pix[tname]['ci_hi']:.3f}]",
                      "p":"--", "Sig":"--"})
    for mname in MODEL_NAMES:
        for tname in TARGET_NAMES:
            r = leak[mname][tname]
            rows1.append({"Dataset":ds_name, "Model":mname, "Target":tname,
                          "AUROC":f"{r['auroc']:.3f}",
                          "95% CI":f"[{r['ci_lo']:.3f},{r['ci_hi']:.3f}]",
                          "p":f"{r['p']:.4f}", "Sig":r["sig"]})
df1 = pd.DataFrame(rows1)
df1.to_csv(OUTDIR/"tables"/"table1_leakage_auroc.csv", index=False)
write_booktabs(OUTDIR/"tables"/"table1_leakage_auroc.tex", df1,
    f"Demographic leakage AUROC across all attached datasets. "
    f"Sig reflects Benjamini-Hochberg FDR correction at q={FDR_Q} (per dataset, 15 tests each).",
    "tab:leakage", "lllrlrl", group_col="Dataset")

# ── Table 2: H2 paired differences, all datasets ─────────────────────────────
rows2 = []
for ds_name in DATASET_NAMES:
    h2 = H2_RESULTS_BY_DS[ds_name]
    if not h2: continue
    for tname in TARGET_NAMES:
        is_demo = tname in ("sex", "age")
        for mname, d in h2[tname].items():
            fdr_tag = ("FDR-SIG" if d.get("fdr_sig") else "ns (post-FDR)") if is_demo \
                     else "context only"
            rows2.append({"Dataset":ds_name, "Target":tname, "Comparison":f"BioMedCLIP - {mname}",
                          "Delta AUROC":f"{d['mean']:+.3f}",
                          "95% CI":f"[{d['ci_lo']:+.3f},{d['ci_hi']:+.3f}]",
                          "p":f"{d['p']:.4f}", "Sig":fdr_tag})
df2 = pd.DataFrame(rows2) if rows2 else pd.DataFrame(
    columns=["Dataset","Target","Comparison","Delta AUROC","95% CI","p","Sig"])
df2.to_csv(OUTDIR/"tables"/"table2_h2_comparisons.csv", index=False)
write_booktabs(OUTDIR/"tables"/"table2_h2_comparisons.tex", df2,
    f"H2: paired-bootstrap AUROC difference, BioMedCLIP vs. natural-pretrained baselines, "
    f"across all attached datasets. Sig reflects FDR correction at q={FDR_Q} across the 8 "
    f"sex/age tests per dataset; disease rows are context only.",
    "tab:h2", "lllrlrl", group_col="Dataset")

# ── Table 3: ECE gap (H3a), all datasets ─────────────────────────────────────
rows3 = []
for ds_name in DATASET_NAMES:
    calib = CALIB_RES_BY_DS[ds_name]
    for mname in MODEL_NAMES:
        for axis_name in ["sex", "age"]:
            r = calib[mname][axis_name]
            rows3.append({"Dataset":ds_name, "Model":mname, "Axis":axis_name,
                          "Raw Gap":f"{r['raw_gap']:.4f}",
                          "Matched Gap":f"{r['matched_gap']:.4f}",
                          "Prev A/B":f"{r['prev_A']:.2f}/{r['prev_B']:.2f}",
                          "p":f"{r['p']:.4f}", "Sig":r["sig"]})
df3 = pd.DataFrame(rows3)
df3.to_csv(OUTDIR/"tables"/"table3_calibration_gap.csv", index=False)
write_booktabs(OUTDIR/"tables"/"table3_calibration_gap.tex", df3,
    f"H3a: cross-demographic calibration gap, raw vs. prevalence-matched, across all "
    f"attached datasets. Sig reflects FDR correction at q={FDR_Q} (per dataset, "
    f"{len(MODEL_NAMES)*2} tests each).",
    "tab:calib", "lllrrrrl", group_col="Dataset")

# ── Table 4: H4 ablation results (NIH) ───────────────────────────────────────
rows4 = []
for mname in MODEL_NAMES:
    r = H4_RESULTS[mname]
    rows4.append({"Model":mname, "Directions removed":r["n_directions"],
                  "AUROC before":f"{r['auroc_before']:.3f}", "AUROC after":f"{r['auroc_after']:.3f}",
                  "Sanity OK":"Y" if r["sanity_ok"] else "N",
                  "Gap before":f"{r['gap_before']:.4f}", "Gap after":f"{r['gap_after']:.4f}",
                  "Change":f"{r['gap_change']:+.4f} ({r['gap_pct_change']:+.0f}%)"})
df4 = pd.DataFrame(rows4)
df4.to_csv(OUTDIR/"tables"/"table4_h4_ablation.csv", index=False)
write_booktabs(OUTDIR/"tables"/"table4_h4_ablation.tex", df4,
    "H4: age-axis calibration gap before and after iterative-nullspace-projection "
    "ablation of the age-decoding direction(s) (NIH).",
    "tab:h4", "lrrrlrrl")

# ── Table 5: Jackknife robustness on H3b ─────────────────────────────────────
df5 = pd.DataFrame([{"Excluded model":r["excluded_model"], "rho":f"{r['rho']:.3f}",
                     "p":f"{r['p']:.3f}", "n":r["n_remaining"]} for r in JACKKNIFE_RESULTS])
df5.to_csv(OUTDIR/"tables"/"table5_jackknife.csv", index=False)
write_booktabs(OUTDIR/"tables"/"table5_jackknife.tex", df5,
    f"Leave-one-model-out jackknife on the H3b correlation (full-sample rho={H3B_RHO:.3f}).",
    "tab:jackknife", "lrrr")

# ── Table 6: Age dose-response (NIH) ─────────────────────────────────────────
rows6 = []
for mname in MODEL_NAMES:
    for b in DOSE_RESPONSE_RESULTS[mname]:
        rows6.append({"Model":mname, "Quantile":b["quantile"],
                      "Age range":f"{b['age_lo']:.0f}-{b['age_hi']:.0f}", "n":b["n"],
                      "ECE":f"{b['ece']:.4f}", "Gap vs. lowest":f"{b['gap_vs_lowest_quantile']:+.4f}"})
df6 = pd.DataFrame(rows6)
df6.to_csv(OUTDIR/"tables"/"table6_dose_response.csv", index=False)
write_booktabs(OUTDIR/"tables"/"table6_dose_response.tex", df6,
    "Age dose-response: ECE per age quantile, per model (NIH).",
    "tab:dose", "lrlrrr", group_col="Model")

# ── Table 7: Multi-seed stability ────────────────────────────────────────────
rows7 = []
for mname in MODEL_NAMES:
    for axis_name in ["sex", "age"]:
        s = MULTISEED_RESULTS[mname][axis_name]
        cv = s["std"]/s["mean"] if s["mean"] > 1e-9 else float("nan")
        rows7.append({"Model":mname, "Axis":axis_name, "Mean gap":f"{s['mean']:.4f}",
                      "Std":f"{s['std']:.4f}", "Range":f"[{s['min']:.4f},{s['max']:.4f}]",
                      "CV":f"{cv:.2f}"})
df7 = pd.DataFrame(rows7)
df7.to_csv(OUTDIR/"tables"/"table7_multiseed_stability.csv", index=False)
write_booktabs(OUTDIR/"tables"/"table7_multiseed_stability.tex", df7,
    f"Multi-seed ({N_MATCH_SEEDS} seeds) stability of prevalence-matched calibration gaps (NIH).",
    "tab:multiseed", "llrrlr", group_col="Model")

print("Tables saved: table1_leakage_auroc, table2_h2_comparisons, table3_calibration_gap,")
print("              table4_h4_ablation, table5_jackknife, table6_dose_response, "
      "table7_multiseed_stability")
print()
print(df3.to_string(index=False))


In [ ]:
# =============================================================================
# CELL 18 — Final Summary, Packaging, and Download
# =============================================================================
print("Generated files:\n")
total_kb = 0.0
for p in sorted(OUTDIR.rglob("*")):
    if p.is_file():
        kb = p.stat().st_size / 1024
        total_kb += kb
        print(f"  {str(p.relative_to(OUTDIR)):<55s}  {kb:6.1f} KB")
print(f"\n  Total: {total_kb/1024:.2f} MB")

shutil.make_archive("outputs", "zip", "outputs")
zip_kb = os.path.getsize("outputs.zip") / 1024
print(f"\nZIP: outputs.zip  ({zip_kb:.0f} KB)")

print("\n" + "="*68)
print("  DEMOGRAPHIC LEAKAGE — FINAL RESULTS SUMMARY (v4)")
print("="*68)

for ds_name in DATASET_NAMES:
    leak, h2 = LEAKAGE_RES_BY_DS[ds_name], H2_RESULTS_BY_DS[ds_name]
    calib = CALIB_RES_BY_DS[ds_name]
    print(f"\n--- {ds_name} ---")
    print(f"  H1: ", end="")
    n_h1 = sum(leak[m][t]["fdr_sig"] for m in MODEL_NAMES for t in TARGET_NAMES)
    print(f"{n_h1}/{len(MODEL_NAMES)*3} tests FDR-significant")
    if h2:
        n_h2 = sum(h2[t][m].get("fdr_sig", False) for t in ("sex","age")
                  for m in h2[t])
        print(f"  H2: {n_h2}/{(len(MODEL_NAMES)-1)*2} demographic comparisons FDR-significant")
    n_h3a = sum(calib[m][a]["fdr_sig"] for m in MODEL_NAMES for a in ("sex","age"))
    print(f"  H3a: {n_h3a}/{len(MODEL_NAMES)*2} calibration tests FDR-significant")

print(f"\n--- H3b (EXPLORATORY, {H3B_AXIS} axis) ---")
for ds_name, res in H3B_BY_DS.items():
    print(f"  {ds_name}: rho={res['rho']:.2f}  p={res['p']:.2f}  "
          f"({res['n_sig']}/{len(MODEL_NAMES)} inputs individually FDR-significant)")

print(f"\n--- H4 (Ablation, NIH) ---")
n_shrunk = sum(1 for r in H4_RESULTS.values() if r["gap_change"] < 0)
n_sane   = sum(1 for r in H4_RESULTS.values() if r["sanity_ok"])
print(f"  Calibration gap shrank in {n_shrunk}/{len(MODEL_NAMES)} models after ablation")
print(f"  Sanity check (residual AUROC near chance) passed in {n_sane}/{len(MODEL_NAMES)} models")

print(f"\n--- Jackknife (NIH) ---")
rhos_jk = [r["rho"] for r in JACKKNIFE_RESULTS if not np.isnan(r["rho"])]
if rhos_jk:
    print(f"  Full rho={H3B_RHO:.2f}, jackknife range=[{min(rhos_jk):.2f},{max(rhos_jk):.2f}]")

print(f"\n--- Dose-response (NIH) ---")
mono_count = sum(1 for bins in DOSE_RESPONSE_RESULTS.values()
                 if all(bins[i]["ece"] <= bins[i+1]["ece"] + 1e-9 for i in range(len(bins)-1)))
print(f"  ECE monotonically increasing with age quantile in {mono_count}/{len(MODEL_NAMES)} models")

print(f"\n--- Multi-seed stability (NIH, ResNet-50 age axis) ---")
if "ResNet-50" in MULTISEED_RESULTS:
    r = MULTISEED_RESULTS["ResNet-50"]["age"]
    cv50 = r["std"]/max(r["mean"],1e-9)
    print(f"  Mean={r['mean']:.4f}  CV={cv50:.2f}  {'STABLE' if cv50 < 0.25 else 'UNSTABLE'}")

print("\n" + "="*68)
print("HONEST BOTTOM LINE:")
print(f"  H1, H2 robustly FDR-significant on NIH" +
      (" and replicate on CheXpert." if CHEXPERT_AVAILABLE else " (CheXpert not attached -- single-dataset only)."))
print(f"  H3a age-axis gap FDR-significant" +
      (", replicates on CheXpert." if CHEXPERT_AVAILABLE else " on NIH (single dataset)."))
print(f"  H4 ablation: {'supports' if n_shrunk >= 3 else 'does NOT clearly support'} a causal "
      f"reading of the leakage-to-calibration link (gap shrank in {n_shrunk}/{len(MODEL_NAMES)} models).")
print("  H3b, jackknife, dose-response, and multi-seed results are robustness/mechanism")
print("  evidence -- report them as such, not as primary FDR-corrected claims.")
print("="*68)

try:
    from google.colab import files
    files.download("outputs.zip")
    print("\nDownload triggered.")
except ImportError:
    print(f"\nNot in Colab -- ZIP at: {Path('outputs.zip').resolve()}")
